# Banking — Python Exception Handling (100 Fully Solved Questions)

A fully solved, end-to-end practice notebook grounded in accounts, deposits, withdrawals, transfers, cards, loans, KYC, fraud monitoring, reconciliation, and audit trails.

**Structure:** 30 easy + 40 medium + 30 difficult = **100 questions**.


## Learning goals

Use `try`, `except`, `else`, `finally`, custom exceptions, chaining, validation, retries, logging, context managers, and safe batch processing in banking workflows. Each solution is runnable and followed by a short explanation. No real customer data, credentials, PINs, or complete card numbers are used anywhere.


---

## Easy

Fundamentals and direct applications.


### Q1. Parse a deposit amount for banking case 1; use sample reference `ACC-1001` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q1: Parse a deposit amount (ACC-1001)
def parse_deposit_amount(raw_amount):
    """Parse a user-supplied deposit amount string into a positive float.

    Narrow exception: float() raises ValueError on bad text, not on business
    rules -- so we catch ValueError specifically, then apply the business
    rule (amount must be > 0) as a separate, explicit check.
    """
    try:
        amount = float(raw_amount)
    except ValueError as exc:
        raise ValueError(f"'{raw_amount}' is not a valid deposit amount") from exc

    if amount <= 0:
        raise ValueError("Deposit amount must be greater than zero")
    return amount


raw_input_value = "-40"  # sample deposit input for ACC-1001
try:
    amount = parse_deposit_amount(raw_input_value)
    print(f"[ACC-1001] Deposit accepted: {amount:.2f}")
except ValueError as e:
    print(f"[ACC-1001] Deposit rejected -> {e}")

# Edge cases
for edge in ["", "12.50", "-5", "NaN", None]:
    try:
        print(f"  edge={edge!r} -> {parse_deposit_amount(edge):.2f}")
    except (ValueError, TypeError) as e:
        print(f"  edge={edge!r} -> rejected: {e}")


**Explanation:** The risky statement is isolated inside `try`, a narrow exception type names the expected failure, and the surrounding flow keeps running with a clear, business-readable message instead of a raw traceback.

### Q2. Handle a missing account key for banking case 1; use sample reference `ACC-1002` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q2: Handle a missing account key (ACC-1002)
accounts = {
    "ACC-1001": {"balance": 1000.0},
    "ACC-2002": {"balance": 500.0},
}

def get_balance(account_ref):
    """KeyError is a programming-level lookup failure; we translate it into
    a domain-specific error so callers only ever need to catch one type."""
    try:
        return accounts[account_ref]["balance"]
    except KeyError as exc:
        raise LookupError(f"Account '{account_ref}' was not found") from exc

for ref in ["ACC-1002", "ACC-1001", ""]:
    try:
        print(f"{ref} balance -> {get_balance(ref):.2f}")
    except LookupError as e:
        print(f"{ref} -> {e}")


**Explanation:** The risky statement is isolated inside `try`, a narrow exception type names the expected failure, and the surrounding flow keeps running with a clear, business-readable message instead of a raw traceback.

### Q3. Prevent division by zero in average balance for banking case 1; use sample reference `ACC-1003` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q3: Prevent division by zero in average balance (ACC-1003)
def average_balance(daily_balances):
    """ZeroDivisionError is an arithmetic/programming failure -- we guard it
    explicitly rather than letting it leak, since 'no data yet' is a normal,
    expected business state (a brand-new account), not an error."""
    try:
        return sum(daily_balances) / len(daily_balances)
    except ZeroDivisionError:
        return 0.0

sample = [100.0, 250.0]
print(f"[ACC-1003] Average balance -> {average_balance(sample):.2f}")
print(f"[ACC-1003] Average balance (no history) -> {average_balance([]):.2f}")


**Explanation:** The risky statement is isolated inside `try`, a narrow exception type names the expected failure, and the surrounding flow keeps running with a clear, business-readable message instead of a raw traceback.

### Q4. Open a transaction file safely for banking case 1; use sample reference `ACC-1004` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q4: Open a transaction file safely (ACC-1004)
import os

def read_transaction_log(path):
    """OSError/FileNotFoundError is an infrastructure failure. We catch it
    narrowly and surface a clear message, keeping cleanup guaranteed."""
    fh = None
    try:
        fh = open(path, "r", encoding="utf-8")
        return fh.read()
    except FileNotFoundError as exc:
        raise FileNotFoundError(f"Transaction log for ACC-1004 not found at '{path}'") from exc
    finally:
        if fh is not None:
            fh.close()
        print(f"[ACC-1004] file handle cleanup complete")

for path in [f"/tmp/ACC-1004_case1_missing.log"]:
    try:
        read_transaction_log(path)
    except FileNotFoundError as e:
        print(f"[ACC-1004] -> {e}")


**Explanation:** The risky statement is isolated inside `try`, a narrow exception type names the expected failure, and the surrounding flow keeps running with a clear, business-readable message instead of a raw traceback.

### Q5. Reject a negative withdrawal for banking case 1; use sample reference `ACC-1005` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q5: Reject a negative withdrawal (ACC-1005)
class InvalidTransactionError(Exception):
    """Business-rule failure: distinct from ValueError/TypeError bugs."""


def withdraw(balance, amount):
    if not isinstance(amount, (int, float)) or isinstance(amount, bool):
        raise TypeError("Withdrawal amount must be numeric")
    if amount < 0:
        raise InvalidTransactionError(f"[ACC-1005] Withdrawal amount cannot be negative: {amount}")
    if amount > balance:
        raise InvalidTransactionError(f"[ACC-1005] Insufficient balance for withdrawal of {amount}")
    return balance - amount

balance = 200.0
try:
    balance = withdraw(balance, 100)
    print(f"[ACC-1005] New balance -> {balance:.2f}")
except InvalidTransactionError as e:
    print(f"[ACC-1005] Withdrawal rejected -> {e}")
except TypeError as e:
    print(f"[ACC-1005] Bad input -> {e}")


**Explanation:** The risky statement is isolated inside `try`, a narrow exception type names the expected failure, and the surrounding flow keeps running with a clear, business-readable message instead of a raw traceback.

### Q6. Catch invalid tenure input for banking case 1; use sample reference `ACC-1006` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q6: Catch invalid tenure input for a loan (ACC-1006)
def parse_tenure_months(raw_tenure):
    try:
        tenure = int(raw_tenure)
    except (ValueError, TypeError) as exc:
        raise ValueError(f"[ACC-1006] Tenure must be a whole number of months, got {raw_tenure!r}") from exc
    if tenure <= 0:
        raise ValueError(f"[ACC-1006] Tenure must be positive, got {tenure}")
    return tenure

for raw in ["-5", "0", "24"]:
    try:
        print(f"[ACC-1006] tenure={raw!r} -> {parse_tenure_months(raw)} months")
    except ValueError as e:
        print(f"[ACC-1006] tenure={raw!r} -> {e}")


**Explanation:** The risky statement is isolated inside `try`, a narrow exception type names the expected failure, and the surrounding flow keeps running with a clear, business-readable message instead of a raw traceback.

### Q7. Use else after pin-format validation for banking case 1; use sample reference `ACC-1007` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q7: Use else after pin-format validation (ACC-1007)
def validate_pin_format(pin):
    if not (isinstance(pin, str) and pin.isdigit() and len(pin) == 4):
        raise ValueError(f"[ACC-1007] PIN must be exactly 4 digits, got {pin!r}")
    return True

def check_pin(pin):
    try:
        validate_pin_format(pin)
    except ValueError as e:
        print(f"[ACC-1007] format check failed -> {e}")
    else:
        # 'else' only runs when no exception occurred -- keeps the success
        # path separate from the try body so we don't accidentally catch
        # errors raised by the success logic itself.
        print(f"[ACC-1007] PIN format OK, proceeding to authenticate '{pin}'")

check_pin("12a4")
check_pin("0000")


**Explanation:** The risky statement is isolated inside `try`, a narrow exception type names the expected failure, and the surrounding flow keeps running with a clear, business-readable message instead of a raw traceback.

### Q8. Use finally to close a session for banking case 1; use sample reference `ACC-1008` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q8: Use finally to close a session (ACC-1008)
class Session:
    def __init__(self, account_ref):
        self.account_ref = account_ref
        self.open = True
        print(f"[{self.account_ref}] session opened")

    def close(self):
        self.open = False
        print(f"[{self.account_ref}] session closed")

def run_transaction(account_ref, should_fail):
    session = Session(account_ref)
    try:
        if should_fail:
            raise RuntimeError(f"[{account_ref}] simulated processing failure")
        print(f"[{account_ref}] transaction processed")
    finally:
        # finally always runs -- guarantees the session is closed whether
        # or not the transaction succeeded.
        session.close()

try:
    run_transaction("ACC-1008", False)
except RuntimeError as e:
    print(f"[ACC-1008] -> {e}")


**Explanation:** The risky statement is isolated inside `try`, a narrow exception type names the expected failure, and the surrounding flow keeps running with a clear, business-readable message instead of a raw traceback.

### Q9. Catch multiple numeric conversion errors for banking case 1; use sample reference `ACC-1009` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q9: Catch multiple numeric conversion errors (ACC-1009)
def to_amount(raw):
    """float() can raise ValueError (bad text) or TypeError (bad type, e.g.
    None). Both are 'this input cannot become a number' -- so it is correct
    to group them in one except clause rather than duplicating logic."""
    try:
        value = float(raw)
    except (ValueError, TypeError) as exc:
        raise ValueError(f"[ACC-1009] Cannot convert {raw!r} to an amount") from exc
    if value != value:  # NaN check
        raise ValueError(f"[ACC-1009] Amount cannot be NaN")
    return value

for raw in ['1e400', 'abc', None, '12.5']:
    try:
        print(f"[ACC-1009] {raw!r} -> {to_amount(raw):.2f}")
    except ValueError as e:
        print(f"[ACC-1009] {raw!r} -> {e}")


**Explanation:** The risky statement is isolated inside `try`, a narrow exception type names the expected failure, and the surrounding flow keeps running with a clear, business-readable message instead of a raw traceback.

### Q10. Raise a clear valueerror for banking case 1; use sample reference `ACC-1010` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q10: Raise a clear ValueError (ACC-1010)
def set_interest_rate(rate_percent):
    """Raise ValueError directly (no prior exception to chain) with a
    message specific enough to act on -- name the field, the bound, and
    the value received."""
    if not isinstance(rate_percent, (int, float)) or isinstance(rate_percent, bool):
        raise ValueError(f"[ACC-1010] interest rate must be numeric, got {type(rate_percent).__name__}")
    if not (0 <= rate_percent <= 25):
        raise ValueError(f"[ACC-1010] interest rate must be between 0 and 25 percent, got {rate_percent}")
    return rate_percent

for rate in [7.5, -1, 40, "bad"]:
    try:
        print(f"[ACC-1010] rate {rate!r} accepted -> {set_interest_rate(rate)}")
    except ValueError as e:
        print(f"[ACC-1010] rate {rate!r} -> {e}")


**Explanation:** The risky statement is isolated inside `try`, a narrow exception type names the expected failure, and the surrounding flow keeps running with a clear, business-readable message instead of a raw traceback.

### Q11. Parse a deposit amount for banking case 2; use sample reference `ACC-1011` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q11: Parse a deposit amount (ACC-1011)
def parse_deposit_amount(raw_amount):
    """Parse a user-supplied deposit amount string into a positive float.

    Narrow exception: float() raises ValueError on bad text, not on business
    rules -- so we catch ValueError specifically, then apply the business
    rule (amount must be > 0) as a separate, explicit check.
    """
    try:
        amount = float(raw_amount)
    except ValueError as exc:
        raise ValueError(f"'{raw_amount}' is not a valid deposit amount") from exc

    if amount <= 0:
        raise ValueError("Deposit amount must be greater than zero")
    return amount


raw_input_value = "abc"  # sample deposit input for ACC-1011
try:
    amount = parse_deposit_amount(raw_input_value)
    print(f"[ACC-1011] Deposit accepted: {amount:.2f}")
except ValueError as e:
    print(f"[ACC-1011] Deposit rejected -> {e}")

# Edge cases
for edge in ["", "12.50", "-5", "NaN", None]:
    try:
        print(f"  edge={edge!r} -> {parse_deposit_amount(edge):.2f}")
    except (ValueError, TypeError) as e:
        print(f"  edge={edge!r} -> rejected: {e}")


**Explanation:** The risky statement is isolated inside `try`, a narrow exception type names the expected failure, and the surrounding flow keeps running with a clear, business-readable message instead of a raw traceback.

### Q12. Handle a missing account key for banking case 2; use sample reference `ACC-1012` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q12: Handle a missing account key (ACC-1012)
accounts = {
    "ACC-1001": {"balance": 1000.0},
    "ACC-2002": {"balance": 500.0},
}

def get_balance(account_ref):
    """KeyError is a programming-level lookup failure; we translate it into
    a domain-specific error so callers only ever need to catch one type."""
    try:
        return accounts[account_ref]["balance"]
    except KeyError as exc:
        raise LookupError(f"Account '{account_ref}' was not found") from exc

for ref in ["ACC-1012", "ACC-1001", ""]:
    try:
        print(f"{ref} balance -> {get_balance(ref):.2f}")
    except LookupError as e:
        print(f"{ref} -> {e}")


**Explanation:** The risky statement is isolated inside `try`, a narrow exception type names the expected failure, and the surrounding flow keeps running with a clear, business-readable message instead of a raw traceback.

### Q13. Prevent division by zero in average balance for banking case 2; use sample reference `ACC-1013` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q13: Prevent division by zero in average balance (ACC-1013)
def average_balance(daily_balances):
    """ZeroDivisionError is an arithmetic/programming failure -- we guard it
    explicitly rather than letting it leak, since 'no data yet' is a normal,
    expected business state (a brand-new account), not an error."""
    try:
        return sum(daily_balances) / len(daily_balances)
    except ZeroDivisionError:
        return 0.0

sample = []
print(f"[ACC-1013] Average balance -> {average_balance(sample):.2f}")
print(f"[ACC-1013] Average balance (no history) -> {average_balance([]):.2f}")


**Explanation:** The risky statement is isolated inside `try`, a narrow exception type names the expected failure, and the surrounding flow keeps running with a clear, business-readable message instead of a raw traceback.

### Q14. Open a transaction file safely for banking case 2; use sample reference `ACC-1014` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q14: Open a transaction file safely (ACC-1014)
import os

def read_transaction_log(path):
    """OSError/FileNotFoundError is an infrastructure failure. We catch it
    narrowly and surface a clear message, keeping cleanup guaranteed."""
    fh = None
    try:
        fh = open(path, "r", encoding="utf-8")
        return fh.read()
    except FileNotFoundError as exc:
        raise FileNotFoundError(f"Transaction log for ACC-1014 not found at '{path}'") from exc
    finally:
        if fh is not None:
            fh.close()
        print(f"[ACC-1014] file handle cleanup complete")

for path in [f"/tmp/ACC-1014_case2_missing.log"]:
    try:
        read_transaction_log(path)
    except FileNotFoundError as e:
        print(f"[ACC-1014] -> {e}")


**Explanation:** The risky statement is isolated inside `try`, a narrow exception type names the expected failure, and the surrounding flow keeps running with a clear, business-readable message instead of a raw traceback.

### Q15. Reject a negative withdrawal for banking case 2; use sample reference `ACC-1015` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q15: Reject a negative withdrawal (ACC-1015)
class InvalidTransactionError(Exception):
    """Business-rule failure: distinct from ValueError/TypeError bugs."""


def withdraw(balance, amount):
    if not isinstance(amount, (int, float)) or isinstance(amount, bool):
        raise TypeError("Withdrawal amount must be numeric")
    if amount < 0:
        raise InvalidTransactionError(f"[ACC-1015] Withdrawal amount cannot be negative: {amount}")
    if amount > balance:
        raise InvalidTransactionError(f"[ACC-1015] Insufficient balance for withdrawal of {amount}")
    return balance - amount

balance = 200.0
try:
    balance = withdraw(balance, -0.01)
    print(f"[ACC-1015] New balance -> {balance:.2f}")
except InvalidTransactionError as e:
    print(f"[ACC-1015] Withdrawal rejected -> {e}")
except TypeError as e:
    print(f"[ACC-1015] Bad input -> {e}")


**Explanation:** The risky statement is isolated inside `try`, a narrow exception type names the expected failure, and the surrounding flow keeps running with a clear, business-readable message instead of a raw traceback.

### Q16. Catch invalid tenure input for banking case 2; use sample reference `ACC-1016` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q16: Catch invalid tenure input for a loan (ACC-1016)
def parse_tenure_months(raw_tenure):
    try:
        tenure = int(raw_tenure)
    except (ValueError, TypeError) as exc:
        raise ValueError(f"[ACC-1016] Tenure must be a whole number of months, got {raw_tenure!r}") from exc
    if tenure <= 0:
        raise ValueError(f"[ACC-1016] Tenure must be positive, got {tenure}")
    return tenure

for raw in ["twelve", "0", "24"]:
    try:
        print(f"[ACC-1016] tenure={raw!r} -> {parse_tenure_months(raw)} months")
    except ValueError as e:
        print(f"[ACC-1016] tenure={raw!r} -> {e}")


**Explanation:** The risky statement is isolated inside `try`, a narrow exception type names the expected failure, and the surrounding flow keeps running with a clear, business-readable message instead of a raw traceback.

### Q17. Use else after pin-format validation for banking case 2; use sample reference `ACC-1017` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q17: Use else after pin-format validation (ACC-1017)
def validate_pin_format(pin):
    if not (isinstance(pin, str) and pin.isdigit() and len(pin) == 4):
        raise ValueError(f"[ACC-1017] PIN must be exactly 4 digits, got {pin!r}")
    return True

def check_pin(pin):
    try:
        validate_pin_format(pin)
    except ValueError as e:
        print(f"[ACC-1017] format check failed -> {e}")
    else:
        # 'else' only runs when no exception occurred -- keeps the success
        # path separate from the try body so we don't accidentally catch
        # errors raised by the success logic itself.
        print(f"[ACC-1017] PIN format OK, proceeding to authenticate '{pin}'")

check_pin("12345")
check_pin("0000")


**Explanation:** The risky statement is isolated inside `try`, a narrow exception type names the expected failure, and the surrounding flow keeps running with a clear, business-readable message instead of a raw traceback.

### Q18. Use finally to close a session for banking case 2; use sample reference `ACC-1018` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q18: Use finally to close a session (ACC-1018)
class Session:
    def __init__(self, account_ref):
        self.account_ref = account_ref
        self.open = True
        print(f"[{self.account_ref}] session opened")

    def close(self):
        self.open = False
        print(f"[{self.account_ref}] session closed")

def run_transaction(account_ref, should_fail):
    session = Session(account_ref)
    try:
        if should_fail:
            raise RuntimeError(f"[{account_ref}] simulated processing failure")
        print(f"[{account_ref}] transaction processed")
    finally:
        # finally always runs -- guarantees the session is closed whether
        # or not the transaction succeeded.
        session.close()

try:
    run_transaction("ACC-1018", True)
except RuntimeError as e:
    print(f"[ACC-1018] -> {e}")


**Explanation:** The risky statement is isolated inside `try`, a narrow exception type names the expected failure, and the surrounding flow keeps running with a clear, business-readable message instead of a raw traceback.

### Q19. Catch multiple numeric conversion errors for banking case 2; use sample reference `ACC-1019` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q19: Catch multiple numeric conversion errors (ACC-1019)
def to_amount(raw):
    """float() can raise ValueError (bad text) or TypeError (bad type, e.g.
    None). Both are 'this input cannot become a number' -- so it is correct
    to group them in one except clause rather than duplicating logic."""
    try:
        value = float(raw)
    except (ValueError, TypeError) as exc:
        raise ValueError(f"[ACC-1019] Cannot convert {raw!r} to an amount") from exc
    if value != value:  # NaN check
        raise ValueError(f"[ACC-1019] Amount cannot be NaN")
    return value

for raw in ['abc', None, '12.5', '1e400']:
    try:
        print(f"[ACC-1019] {raw!r} -> {to_amount(raw):.2f}")
    except ValueError as e:
        print(f"[ACC-1019] {raw!r} -> {e}")


**Explanation:** The risky statement is isolated inside `try`, a narrow exception type names the expected failure, and the surrounding flow keeps running with a clear, business-readable message instead of a raw traceback.

### Q20. Raise a clear valueerror for banking case 2; use sample reference `ACC-1020` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q20: Raise a clear ValueError (ACC-1020)
def set_interest_rate(rate_percent):
    """Raise ValueError directly (no prior exception to chain) with a
    message specific enough to act on -- name the field, the bound, and
    the value received."""
    if not isinstance(rate_percent, (int, float)) or isinstance(rate_percent, bool):
        raise ValueError(f"[ACC-1020] interest rate must be numeric, got {type(rate_percent).__name__}")
    if not (0 <= rate_percent <= 25):
        raise ValueError(f"[ACC-1020] interest rate must be between 0 and 25 percent, got {rate_percent}")
    return rate_percent

for rate in [7.5, -1, 40, "bad"]:
    try:
        print(f"[ACC-1020] rate {rate!r} accepted -> {set_interest_rate(rate)}")
    except ValueError as e:
        print(f"[ACC-1020] rate {rate!r} -> {e}")


**Explanation:** The risky statement is isolated inside `try`, a narrow exception type names the expected failure, and the surrounding flow keeps running with a clear, business-readable message instead of a raw traceback.

### Q21. Parse a deposit amount for banking case 3; use sample reference `ACC-1021` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q21: Parse a deposit amount (ACC-1021)
def parse_deposit_amount(raw_amount):
    """Parse a user-supplied deposit amount string into a positive float.

    Narrow exception: float() raises ValueError on bad text, not on business
    rules -- so we catch ValueError specifically, then apply the business
    rule (amount must be > 0) as a separate, explicit check.
    """
    try:
        amount = float(raw_amount)
    except ValueError as exc:
        raise ValueError(f"'{raw_amount}' is not a valid deposit amount") from exc

    if amount <= 0:
        raise ValueError("Deposit amount must be greater than zero")
    return amount


raw_input_value = "0"  # sample deposit input for ACC-1021
try:
    amount = parse_deposit_amount(raw_input_value)
    print(f"[ACC-1021] Deposit accepted: {amount:.2f}")
except ValueError as e:
    print(f"[ACC-1021] Deposit rejected -> {e}")

# Edge cases
for edge in ["", "12.50", "-5", "NaN", None]:
    try:
        print(f"  edge={edge!r} -> {parse_deposit_amount(edge):.2f}")
    except (ValueError, TypeError) as e:
        print(f"  edge={edge!r} -> rejected: {e}")


**Explanation:** The risky statement is isolated inside `try`, a narrow exception type names the expected failure, and the surrounding flow keeps running with a clear, business-readable message instead of a raw traceback.

### Q22. Handle a missing account key for banking case 3; use sample reference `ACC-1022` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q22: Handle a missing account key (ACC-1022)
accounts = {
    "ACC-1001": {"balance": 1000.0},
    "ACC-2002": {"balance": 500.0},
}

def get_balance(account_ref):
    """KeyError is a programming-level lookup failure; we translate it into
    a domain-specific error so callers only ever need to catch one type."""
    try:
        return accounts[account_ref]["balance"]
    except KeyError as exc:
        raise LookupError(f"Account '{account_ref}' was not found") from exc

for ref in ["ACC-1022", "ACC-1001", ""]:
    try:
        print(f"{ref} balance -> {get_balance(ref):.2f}")
    except LookupError as e:
        print(f"{ref} -> {e}")


**Explanation:** The risky statement is isolated inside `try`, a narrow exception type names the expected failure, and the surrounding flow keeps running with a clear, business-readable message instead of a raw traceback.

### Q23. Prevent division by zero in average balance for banking case 3; use sample reference `ACC-1023` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q23: Prevent division by zero in average balance (ACC-1023)
def average_balance(daily_balances):
    """ZeroDivisionError is an arithmetic/programming failure -- we guard it
    explicitly rather than letting it leak, since 'no data yet' is a normal,
    expected business state (a brand-new account), not an error."""
    try:
        return sum(daily_balances) / len(daily_balances)
    except ZeroDivisionError:
        return 0.0

sample = [100.0, 250.0]
print(f"[ACC-1023] Average balance -> {average_balance(sample):.2f}")
print(f"[ACC-1023] Average balance (no history) -> {average_balance([]):.2f}")


**Explanation:** The risky statement is isolated inside `try`, a narrow exception type names the expected failure, and the surrounding flow keeps running with a clear, business-readable message instead of a raw traceback.

### Q24. Open a transaction file safely for banking case 3; use sample reference `ACC-1024` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q24: Open a transaction file safely (ACC-1024)
import os

def read_transaction_log(path):
    """OSError/FileNotFoundError is an infrastructure failure. We catch it
    narrowly and surface a clear message, keeping cleanup guaranteed."""
    fh = None
    try:
        fh = open(path, "r", encoding="utf-8")
        return fh.read()
    except FileNotFoundError as exc:
        raise FileNotFoundError(f"Transaction log for ACC-1024 not found at '{path}'") from exc
    finally:
        if fh is not None:
            fh.close()
        print(f"[ACC-1024] file handle cleanup complete")

for path in [f"/tmp/ACC-1024_case3_missing.log"]:
    try:
        read_transaction_log(path)
    except FileNotFoundError as e:
        print(f"[ACC-1024] -> {e}")


**Explanation:** The risky statement is isolated inside `try`, a narrow exception type names the expected failure, and the surrounding flow keeps running with a clear, business-readable message instead of a raw traceback.

### Q25. Reject a negative withdrawal for banking case 3; use sample reference `ACC-1025` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q25: Reject a negative withdrawal (ACC-1025)
class InvalidTransactionError(Exception):
    """Business-rule failure: distinct from ValueError/TypeError bugs."""


def withdraw(balance, amount):
    if not isinstance(amount, (int, float)) or isinstance(amount, bool):
        raise TypeError("Withdrawal amount must be numeric")
    if amount < 0:
        raise InvalidTransactionError(f"[ACC-1025] Withdrawal amount cannot be negative: {amount}")
    if amount > balance:
        raise InvalidTransactionError(f"[ACC-1025] Insufficient balance for withdrawal of {amount}")
    return balance - amount

balance = 200.0
try:
    balance = withdraw(balance, -50)
    print(f"[ACC-1025] New balance -> {balance:.2f}")
except InvalidTransactionError as e:
    print(f"[ACC-1025] Withdrawal rejected -> {e}")
except TypeError as e:
    print(f"[ACC-1025] Bad input -> {e}")


**Explanation:** The risky statement is isolated inside `try`, a narrow exception type names the expected failure, and the surrounding flow keeps running with a clear, business-readable message instead of a raw traceback.

### Q26. Catch invalid tenure input for banking case 3; use sample reference `ACC-1026` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q26: Catch invalid tenure input for a loan (ACC-1026)
def parse_tenure_months(raw_tenure):
    try:
        tenure = int(raw_tenure)
    except (ValueError, TypeError) as exc:
        raise ValueError(f"[ACC-1026] Tenure must be a whole number of months, got {raw_tenure!r}") from exc
    if tenure <= 0:
        raise ValueError(f"[ACC-1026] Tenure must be positive, got {tenure}")
    return tenure

for raw in ["36", "0", "24"]:
    try:
        print(f"[ACC-1026] tenure={raw!r} -> {parse_tenure_months(raw)} months")
    except ValueError as e:
        print(f"[ACC-1026] tenure={raw!r} -> {e}")


**Explanation:** The risky statement is isolated inside `try`, a narrow exception type names the expected failure, and the surrounding flow keeps running with a clear, business-readable message instead of a raw traceback.

### Q27. Use else after pin-format validation for banking case 3; use sample reference `ACC-1027` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q27: Use else after pin-format validation (ACC-1027)
def validate_pin_format(pin):
    if not (isinstance(pin, str) and pin.isdigit() and len(pin) == 4):
        raise ValueError(f"[ACC-1027] PIN must be exactly 4 digits, got {pin!r}")
    return True

def check_pin(pin):
    try:
        validate_pin_format(pin)
    except ValueError as e:
        print(f"[ACC-1027] format check failed -> {e}")
    else:
        # 'else' only runs when no exception occurred -- keeps the success
        # path separate from the try body so we don't accidentally catch
        # errors raised by the success logic itself.
        print(f"[ACC-1027] PIN format OK, proceeding to authenticate '{pin}'")

check_pin("1234")
check_pin("0000")


**Explanation:** The risky statement is isolated inside `try`, a narrow exception type names the expected failure, and the surrounding flow keeps running with a clear, business-readable message instead of a raw traceback.

### Q28. Use finally to close a session for banking case 3; use sample reference `ACC-1028` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q28: Use finally to close a session (ACC-1028)
class Session:
    def __init__(self, account_ref):
        self.account_ref = account_ref
        self.open = True
        print(f"[{self.account_ref}] session opened")

    def close(self):
        self.open = False
        print(f"[{self.account_ref}] session closed")

def run_transaction(account_ref, should_fail):
    session = Session(account_ref)
    try:
        if should_fail:
            raise RuntimeError(f"[{account_ref}] simulated processing failure")
        print(f"[{account_ref}] transaction processed")
    finally:
        # finally always runs -- guarantees the session is closed whether
        # or not the transaction succeeded.
        session.close()

try:
    run_transaction("ACC-1028", False)
except RuntimeError as e:
    print(f"[ACC-1028] -> {e}")


**Explanation:** The risky statement is isolated inside `try`, a narrow exception type names the expected failure, and the surrounding flow keeps running with a clear, business-readable message instead of a raw traceback.

### Q29. Catch multiple numeric conversion errors for banking case 3; use sample reference `ACC-1029` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q29: Catch multiple numeric conversion errors (ACC-1029)
def to_amount(raw):
    """float() can raise ValueError (bad text) or TypeError (bad type, e.g.
    None). Both are 'this input cannot become a number' -- so it is correct
    to group them in one except clause rather than duplicating logic."""
    try:
        value = float(raw)
    except (ValueError, TypeError) as exc:
        raise ValueError(f"[ACC-1029] Cannot convert {raw!r} to an amount") from exc
    if value != value:  # NaN check
        raise ValueError(f"[ACC-1029] Amount cannot be NaN")
    return value

for raw in [None, '12.5', '1e400', 'abc']:
    try:
        print(f"[ACC-1029] {raw!r} -> {to_amount(raw):.2f}")
    except ValueError as e:
        print(f"[ACC-1029] {raw!r} -> {e}")


**Explanation:** The risky statement is isolated inside `try`, a narrow exception type names the expected failure, and the surrounding flow keeps running with a clear, business-readable message instead of a raw traceback.

### Q30. Raise a clear valueerror for banking case 3; use sample reference `ACC-1030` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q30: Raise a clear ValueError (ACC-1030)
def set_interest_rate(rate_percent):
    """Raise ValueError directly (no prior exception to chain) with a
    message specific enough to act on -- name the field, the bound, and
    the value received."""
    if not isinstance(rate_percent, (int, float)) or isinstance(rate_percent, bool):
        raise ValueError(f"[ACC-1030] interest rate must be numeric, got {type(rate_percent).__name__}")
    if not (0 <= rate_percent <= 25):
        raise ValueError(f"[ACC-1030] interest rate must be between 0 and 25 percent, got {rate_percent}")
    return rate_percent

for rate in [7.5, -1, 40, "bad"]:
    try:
        print(f"[ACC-1030] rate {rate!r} accepted -> {set_interest_rate(rate)}")
    except ValueError as e:
        print(f"[ACC-1030] rate {rate!r} -> {e}")


**Explanation:** The risky statement is isolated inside `try`, a narrow exception type names the expected failure, and the surrounding flow keeps running with a clear, business-readable message instead of a raw traceback.


---

## Medium

Reusable components and multi-step workflows.


### Q31. Create insufficientfundserror for banking case 1; use sample reference `ACC-1031` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q31: Create InsufficientFundsError (ACC-1031)
class BankingError(Exception):
    """Base class for all business-rule banking failures."""

class InsufficientFundsError(BankingError):
    def __init__(self, account_ref, balance, requested):
        self.account_ref = account_ref
        self.balance = balance
        self.requested = requested
        super().__init__(
            f"[{account_ref}] insufficient funds: balance={balance:.2f}, requested={requested:.2f}"
        )

def withdraw(account_ref, balance, amount):
    if amount > balance:
        raise InsufficientFundsError(account_ref, balance, amount)
    return balance - amount

try:
    withdraw("ACC-1031", 1000.0, 1500)
except InsufficientFundsError as e:
    print(f"Rejected -> {e}")
    print(f"  shortfall: {e.requested - e.balance:.2f}")


**Explanation:** This solution separates validation from orchestration, catches errors at the boundary that is actually able to act on them, and keeps enough context (account reference, field name, or cause) for support and audit follow-up.

### Q32. Chain invalidtransactionerror from valueerror for banking case 1; use sample reference `ACC-1032` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q32: Chain InvalidTransactionError from ValueError (ACC-1032)
class InvalidTransactionError(Exception):
    pass

def process_transaction_amount(raw_amount):
    try:
        amount = float(raw_amount)
        if amount <= 0:
            raise ValueError("amount must be positive")
    except ValueError as exc:
        # 'raise ... from exc' preserves the original ValueError as
        # __cause__, so the traceback still shows the root parsing failure.
        raise InvalidTransactionError(f"[ACC-1032] invalid transaction amount {raw_amount!r}") from exc
    return amount

try:
    process_transaction_amount("not-a-number")
except InvalidTransactionError as e:
    print(f"{e}")
    print(f"  caused by: {e.__cause__!r}")


**Explanation:** This solution separates validation from orchestration, catches errors at the boundary that is actually able to act on them, and keeps enough context (account reference, field name, or cause) for support and audit follow-up.

### Q33. Process transfers while isolating bad rows for banking case 1; use sample reference `ACC-1033` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q33: Process transfers while isolating bad rows (ACC-1033)
transfers = [
    {"ref": "ACC-1033", "to": "ACC-2001", "amount": 100.0},
    {"ref": "ACC-1033", "to": "ACC-2002", "amount": "bad"},
    {"ref": "ACC-1033", "to": "", "amount": 50.0},
    {"ref": "ACC-1033", "to": "ACC-2003", "amount": -20.0},
]

def validate_transfer(row):
    amount = float(row["amount"])  # ValueError on bad text
    if not row["to"]:
        raise ValueError("missing beneficiary account")
    if amount <= 0:
        raise ValueError("amount must be positive")
    return amount

successes, failures = [], []
for i, row in enumerate(transfers):
    try:
        amount = validate_transfer(row)
        successes.append((i, amount))
    except (ValueError, TypeError) as e:
        # one bad row is isolated and logged; the batch keeps going
        failures.append((i, str(e)))

print(f"[ACC-1033] processed {len(successes)} ok, {len(failures)} failed")
for i, amt in successes:
    print(f"  row {i} OK -> {amt}")
for i, err in failures:
    print(f"  row {i} FAILED -> {err}")


**Explanation:** This solution separates validation from orchestration, catches errors at the boundary that is actually able to act on them, and keeps enough context (account reference, field name, or cause) for support and audit follow-up.

### Q34. Retry a transient payment gateway for banking case 1; use sample reference `ACC-1034` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q34: Retry a transient payment gateway (ACC-1034)
import time

class GatewayTimeoutError(Exception):
    pass

_attempts = {"n": 0}

def call_payment_gateway():
    _attempts["n"] += 1
    if _attempts["n"] <= 1:
        raise GatewayTimeoutError(f"[ACC-1034] gateway timed out (attempt {_attempts['n']})")
    return "PAYMENT-OK"

def call_with_retry(func, max_attempts=3, backoff_seconds=0):
    last_exc = None
    for attempt in range(1, max_attempts + 1):
        try:
            return func()
        except GatewayTimeoutError as e:
            last_exc = e
            print(f"[ACC-1034] attempt {attempt} failed -> {e}")
            time.sleep(backoff_seconds)
    raise GatewayTimeoutError(f"[ACC-1034] gateway unavailable after {max_attempts} attempts") from last_exc

try:
    result = call_with_retry(call_payment_gateway, max_attempts=3)
    print(f"[ACC-1034] result -> {result}")
except GatewayTimeoutError as e:
    print(f"[ACC-1034] gave up -> {e}")


**Explanation:** This solution separates validation from orchestration, catches errors at the boundary that is actually able to act on them, and keeps enough context (account reference, field name, or cause) for support and audit follow-up.

### Q35. Log failed standing instructions for banking case 1; use sample reference `ACC-1035` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q35: Log failed standing instructions (ACC-1035)
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger("standing_instructions")

instructions = [
    {"ref": "ACC-1035", "amount": 200.0},
    {"ref": "ACC-1035", "amount": -50.0},
]

def execute_standing_instruction(instr):
    if instr["amount"] <= 0:
        raise ValueError(f"standing instruction amount must be positive: {instr['amount']}")
    return instr["amount"]

for instr in instructions:
    try:
        amt = execute_standing_instruction(instr)
        logger.info(f"[{instr['ref']}] standing instruction executed: {amt}")
    except ValueError as e:
        # log with context, don't crash the whole batch
        logger.error(f"[{instr['ref']}] standing instruction failed: {e}")


**Explanation:** This solution separates validation from orchestration, catches errors at the boundary that is actually able to act on them, and keeps enough context (account reference, field name, or cause) for support and audit follow-up.

### Q36. Return a structured failure result for banking case 1; use sample reference `ACC-1036` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q36: Return a structured failure result (ACC-1036)
def safe_deposit(account_ref, amount):
    """Instead of letting exceptions propagate to the caller, return a
    structured result. Useful at API/service boundaries where callers
    want a value, not a try/except."""
    try:
        if amount <= 0:
            raise ValueError("deposit amount must be positive")
        return {"ok": True, "account_ref": account_ref, "amount": amount, "error": None}
    except ValueError as e:
        return {"ok": False, "account_ref": account_ref, "amount": None, "error": str(e)}

result = safe_deposit("ACC-1036", -5)
if result["ok"]:
    print(f"[ACC-1036] success -> {result['amount']}")
else:
    print(f"[ACC-1036] failure -> {result['error']}")


**Explanation:** This solution separates validation from orchestration, catches errors at the boundary that is actually able to act on them, and keeps enough context (account reference, field name, or cause) for support and audit follow-up.

### Q37. Validate a nested beneficiary record for banking case 1; use sample reference `ACC-1037` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q37: Validate a nested beneficiary record (ACC-1037)
class KYCValidationError(Exception):
    pass

def validate_beneficiary(record):
    required = ["name", "account", "ifsc"]
    missing = [f for f in required if f not in record or not record[f]]
    if missing:
        raise KYCValidationError(f"[ACC-1037] beneficiary missing fields: {', '.join(missing)}")
    return True

beneficiary = {"name": "Asha Rao", "account": "ACC-9001"}
try:
    validate_beneficiary(beneficiary)
    print(f"[ACC-1037] beneficiary valid -> {beneficiary['name']}")
except KYCValidationError as e:
    print(f"[ACC-1037] -> {e}")


**Explanation:** This solution separates validation from orchestration, catches errors at the boundary that is actually able to act on them, and keeps enough context (account reference, field name, or cause) for support and audit follow-up.

### Q38. Use a context manager for an audit file for banking case 1; use sample reference `ACC-1038` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q38: Use a context manager for an audit file (ACC-1038)
import contextlib, io

@contextlib.contextmanager
def audit_writer(buffer):
    """Guarantees the audit entry is flushed even if the body raises."""
    print(f"[ACC-1038] audit session started")
    try:
        yield buffer
    finally:
        buffer.flush()
        print(f"[ACC-1038] audit session closed, {len(buffer.getvalue())} bytes written")

buf = io.StringIO()
try:
    with audit_writer(buf) as log:
        log.write(f"[ACC-1038] transaction recorded\n")
        raise RuntimeError("downstream notification failed")
except RuntimeError as e:
    print(f"[ACC-1038] handled after cleanup -> {e}")
print(buf.getvalue())


**Explanation:** This solution separates validation from orchestration, catches errors at the boundary that is actually able to act on them, and keeps enough context (account reference, field name, or cause) for support and audit follow-up.

### Q39. Separate validation and recovery for banking case 1; use sample reference `ACC-1039` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q39: Separate validation and recovery (ACC-1039)
class ValidationError(Exception):
    pass

class RecoverableError(Exception):
    pass

def validate_transfer_request(amount, to_account):
    # Validation: caller mistakes -- do NOT retry these, fail fast.
    if amount <= 0:
        raise ValidationError(f"[ACC-1039] amount must be positive")
    if not to_account:
        raise ValidationError(f"[ACC-1039] destination account is required")

def submit_transfer(amount, to_account):
    # Recovery: infrastructure hiccups -- these ARE worth retrying.
    if to_account == "ACC-DOWN":
        raise RecoverableError(f"[ACC-1039] downstream ledger temporarily unavailable")
    return f"TXN-{to_account}-{amount}"

def transfer(amount, to_account, retries=2):
    validate_transfer_request(amount, to_account)  # not retried
    for attempt in range(retries + 1):
        try:
            return submit_transfer(amount, to_account)
        except RecoverableError as e:
            print(f"[ACC-1039] retry {attempt + 1} after -> {e}")
    raise RecoverableError(f"[ACC-1039] transfer failed after {retries} retries")

try:
    print(transfer(100.0, "ACC-2001"))
except ValidationError as e:
    print(f"[ACC-1039] bad request -> {e}")


**Explanation:** This solution separates validation from orchestration, catches errors at the boundary that is actually able to act on them, and keeps enough context (account reference, field name, or cause) for support and audit follow-up.

### Q40. Collect all kyc validation errors for banking case 1; use sample reference `ACC-1040` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q40: Collect all KYC validation errors (ACC-1040)
class KYCValidationError(Exception):
    def __init__(self, errors):
        self.errors = errors
        super().__init__("; ".join(errors))

def validate_kyc(record):
    """Collect every problem instead of stopping at the first, so the
    customer/ops team can fix everything in one pass."""
    errors = []
    if not record.get("pan"):
        errors.append("PAN is required")
    if not record.get("address"):
        errors.append("address is required")
    if not record.get("dob"):
        errors.append("date of birth is required")
    if errors:
        raise KYCValidationError(errors)
    return True

record = {"pan": "ABCDE1234F"} if 1 % 2 else {}
try:
    validate_kyc(record)
    print(f"[ACC-1040] KYC valid")
except KYCValidationError as e:
    print(f"[ACC-1040] KYC failed with {len(e.errors)} issue(s):")
    for err in e.errors:
        print(f"  - {err}")


**Explanation:** This solution separates validation from orchestration, catches errors at the boundary that is actually able to act on them, and keeps enough context (account reference, field name, or cause) for support and audit follow-up.

### Q41. Create insufficientfundserror for banking case 2; use sample reference `ACC-1041` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q41: Create InsufficientFundsError (ACC-1041)
class BankingError(Exception):
    """Base class for all business-rule banking failures."""

class InsufficientFundsError(BankingError):
    def __init__(self, account_ref, balance, requested):
        self.account_ref = account_ref
        self.balance = balance
        self.requested = requested
        super().__init__(
            f"[{account_ref}] insufficient funds: balance={balance:.2f}, requested={requested:.2f}"
        )

def withdraw(account_ref, balance, amount):
    if amount > balance:
        raise InsufficientFundsError(account_ref, balance, amount)
    return balance - amount

try:
    withdraw("ACC-1041", 1000.0, 999.99)
except InsufficientFundsError as e:
    print(f"Rejected -> {e}")
    print(f"  shortfall: {e.requested - e.balance:.2f}")


**Explanation:** This solution separates validation from orchestration, catches errors at the boundary that is actually able to act on them, and keeps enough context (account reference, field name, or cause) for support and audit follow-up.

### Q42. Chain invalidtransactionerror from valueerror for banking case 2; use sample reference `ACC-1042` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q42: Chain InvalidTransactionError from ValueError (ACC-1042)
class InvalidTransactionError(Exception):
    pass

def process_transaction_amount(raw_amount):
    try:
        amount = float(raw_amount)
        if amount <= 0:
            raise ValueError("amount must be positive")
    except ValueError as exc:
        # 'raise ... from exc' preserves the original ValueError as
        # __cause__, so the traceback still shows the root parsing failure.
        raise InvalidTransactionError(f"[ACC-1042] invalid transaction amount {raw_amount!r}") from exc
    return amount

try:
    process_transaction_amount("-1")
except InvalidTransactionError as e:
    print(f"{e}")
    print(f"  caused by: {e.__cause__!r}")


**Explanation:** This solution separates validation from orchestration, catches errors at the boundary that is actually able to act on them, and keeps enough context (account reference, field name, or cause) for support and audit follow-up.

### Q43. Process transfers while isolating bad rows for banking case 2; use sample reference `ACC-1043` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q43: Process transfers while isolating bad rows (ACC-1043)
transfers = [
    {"ref": "ACC-1043", "to": "ACC-2001", "amount": 100.0},
    {"ref": "ACC-1043", "to": "ACC-2002", "amount": "bad"},
    {"ref": "ACC-1043", "to": "", "amount": 50.0},
    {"ref": "ACC-1043", "to": "ACC-2003", "amount": -20.0},
]

def validate_transfer(row):
    amount = float(row["amount"])  # ValueError on bad text
    if not row["to"]:
        raise ValueError("missing beneficiary account")
    if amount <= 0:
        raise ValueError("amount must be positive")
    return amount

successes, failures = [], []
for i, row in enumerate(transfers):
    try:
        amount = validate_transfer(row)
        successes.append((i, amount))
    except (ValueError, TypeError) as e:
        # one bad row is isolated and logged; the batch keeps going
        failures.append((i, str(e)))

print(f"[ACC-1043] processed {len(successes)} ok, {len(failures)} failed")
for i, amt in successes:
    print(f"  row {i} OK -> {amt}")
for i, err in failures:
    print(f"  row {i} FAILED -> {err}")


**Explanation:** This solution separates validation from orchestration, catches errors at the boundary that is actually able to act on them, and keeps enough context (account reference, field name, or cause) for support and audit follow-up.

### Q44. Retry a transient payment gateway for banking case 2; use sample reference `ACC-1044` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q44: Retry a transient payment gateway (ACC-1044)
import time

class GatewayTimeoutError(Exception):
    pass

_attempts = {"n": 0}

def call_payment_gateway():
    _attempts["n"] += 1
    if _attempts["n"] <= 2:
        raise GatewayTimeoutError(f"[ACC-1044] gateway timed out (attempt {_attempts['n']})")
    return "PAYMENT-OK"

def call_with_retry(func, max_attempts=3, backoff_seconds=0):
    last_exc = None
    for attempt in range(1, max_attempts + 1):
        try:
            return func()
        except GatewayTimeoutError as e:
            last_exc = e
            print(f"[ACC-1044] attempt {attempt} failed -> {e}")
            time.sleep(backoff_seconds)
    raise GatewayTimeoutError(f"[ACC-1044] gateway unavailable after {max_attempts} attempts") from last_exc

try:
    result = call_with_retry(call_payment_gateway, max_attempts=3)
    print(f"[ACC-1044] result -> {result}")
except GatewayTimeoutError as e:
    print(f"[ACC-1044] gave up -> {e}")


**Explanation:** This solution separates validation from orchestration, catches errors at the boundary that is actually able to act on them, and keeps enough context (account reference, field name, or cause) for support and audit follow-up.

### Q45. Log failed standing instructions for banking case 2; use sample reference `ACC-1045` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q45: Log failed standing instructions (ACC-1045)
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger("standing_instructions")

instructions = [
    {"ref": "ACC-1045", "amount": 200.0},
    {"ref": "ACC-1045", "amount": -50.0},
]

def execute_standing_instruction(instr):
    if instr["amount"] <= 0:
        raise ValueError(f"standing instruction amount must be positive: {instr['amount']}")
    return instr["amount"]

for instr in instructions:
    try:
        amt = execute_standing_instruction(instr)
        logger.info(f"[{instr['ref']}] standing instruction executed: {amt}")
    except ValueError as e:
        # log with context, don't crash the whole batch
        logger.error(f"[{instr['ref']}] standing instruction failed: {e}")


**Explanation:** This solution separates validation from orchestration, catches errors at the boundary that is actually able to act on them, and keeps enough context (account reference, field name, or cause) for support and audit follow-up.

### Q46. Return a structured failure result for banking case 2; use sample reference `ACC-1046` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q46: Return a structured failure result (ACC-1046)
def safe_deposit(account_ref, amount):
    """Instead of letting exceptions propagate to the caller, return a
    structured result. Useful at API/service boundaries where callers
    want a value, not a try/except."""
    try:
        if amount <= 0:
            raise ValueError("deposit amount must be positive")
        return {"ok": True, "account_ref": account_ref, "amount": amount, "error": None}
    except ValueError as e:
        return {"ok": False, "account_ref": account_ref, "amount": None, "error": str(e)}

result = safe_deposit("ACC-1046", 100)
if result["ok"]:
    print(f"[ACC-1046] success -> {result['amount']}")
else:
    print(f"[ACC-1046] failure -> {result['error']}")


**Explanation:** This solution separates validation from orchestration, catches errors at the boundary that is actually able to act on them, and keeps enough context (account reference, field name, or cause) for support and audit follow-up.

### Q47. Validate a nested beneficiary record for banking case 2; use sample reference `ACC-1047` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q47: Validate a nested beneficiary record (ACC-1047)
class KYCValidationError(Exception):
    pass

def validate_beneficiary(record):
    required = ["name", "account", "ifsc"]
    missing = [f for f in required if f not in record or not record[f]]
    if missing:
        raise KYCValidationError(f"[ACC-1047] beneficiary missing fields: {', '.join(missing)}")
    return True

beneficiary = {"name": "Asha Rao", "account": "ACC-9001", "ifsc": "HDFC0001"}
try:
    validate_beneficiary(beneficiary)
    print(f"[ACC-1047] beneficiary valid -> {beneficiary['name']}")
except KYCValidationError as e:
    print(f"[ACC-1047] -> {e}")


**Explanation:** This solution separates validation from orchestration, catches errors at the boundary that is actually able to act on them, and keeps enough context (account reference, field name, or cause) for support and audit follow-up.

### Q48. Use a context manager for an audit file for banking case 2; use sample reference `ACC-1048` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q48: Use a context manager for an audit file (ACC-1048)
import contextlib, io

@contextlib.contextmanager
def audit_writer(buffer):
    """Guarantees the audit entry is flushed even if the body raises."""
    print(f"[ACC-1048] audit session started")
    try:
        yield buffer
    finally:
        buffer.flush()
        print(f"[ACC-1048] audit session closed, {len(buffer.getvalue())} bytes written")

buf = io.StringIO()
try:
    with audit_writer(buf) as log:
        log.write(f"[ACC-1048] transaction recorded\n")
        raise RuntimeError("downstream notification failed")
except RuntimeError as e:
    print(f"[ACC-1048] handled after cleanup -> {e}")
print(buf.getvalue())


**Explanation:** This solution separates validation from orchestration, catches errors at the boundary that is actually able to act on them, and keeps enough context (account reference, field name, or cause) for support and audit follow-up.

### Q49. Separate validation and recovery for banking case 2; use sample reference `ACC-1049` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q49: Separate validation and recovery (ACC-1049)
class ValidationError(Exception):
    pass

class RecoverableError(Exception):
    pass

def validate_transfer_request(amount, to_account):
    # Validation: caller mistakes -- do NOT retry these, fail fast.
    if amount <= 0:
        raise ValidationError(f"[ACC-1049] amount must be positive")
    if not to_account:
        raise ValidationError(f"[ACC-1049] destination account is required")

def submit_transfer(amount, to_account):
    # Recovery: infrastructure hiccups -- these ARE worth retrying.
    if to_account == "ACC-DOWN":
        raise RecoverableError(f"[ACC-1049] downstream ledger temporarily unavailable")
    return f"TXN-{to_account}-{amount}"

def transfer(amount, to_account, retries=2):
    validate_transfer_request(amount, to_account)  # not retried
    for attempt in range(retries + 1):
        try:
            return submit_transfer(amount, to_account)
        except RecoverableError as e:
            print(f"[ACC-1049] retry {attempt + 1} after -> {e}")
    raise RecoverableError(f"[ACC-1049] transfer failed after {retries} retries")

try:
    print(transfer(100.0, "ACC-2001"))
except ValidationError as e:
    print(f"[ACC-1049] bad request -> {e}")


**Explanation:** This solution separates validation from orchestration, catches errors at the boundary that is actually able to act on them, and keeps enough context (account reference, field name, or cause) for support and audit follow-up.

### Q50. Collect all kyc validation errors for banking case 2; use sample reference `ACC-1050` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q50: Collect all KYC validation errors (ACC-1050)
class KYCValidationError(Exception):
    def __init__(self, errors):
        self.errors = errors
        super().__init__("; ".join(errors))

def validate_kyc(record):
    """Collect every problem instead of stopping at the first, so the
    customer/ops team can fix everything in one pass."""
    errors = []
    if not record.get("pan"):
        errors.append("PAN is required")
    if not record.get("address"):
        errors.append("address is required")
    if not record.get("dob"):
        errors.append("date of birth is required")
    if errors:
        raise KYCValidationError(errors)
    return True

record = {"pan": "ABCDE1234F"} if 2 % 2 else {}
try:
    validate_kyc(record)
    print(f"[ACC-1050] KYC valid")
except KYCValidationError as e:
    print(f"[ACC-1050] KYC failed with {len(e.errors)} issue(s):")
    for err in e.errors:
        print(f"  - {err}")


**Explanation:** This solution separates validation from orchestration, catches errors at the boundary that is actually able to act on them, and keeps enough context (account reference, field name, or cause) for support and audit follow-up.

### Q51. Create insufficientfundserror for banking case 3; use sample reference `ACC-1051` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q51: Create InsufficientFundsError (ACC-1051)
class BankingError(Exception):
    """Base class for all business-rule banking failures."""

class InsufficientFundsError(BankingError):
    def __init__(self, account_ref, balance, requested):
        self.account_ref = account_ref
        self.balance = balance
        self.requested = requested
        super().__init__(
            f"[{account_ref}] insufficient funds: balance={balance:.2f}, requested={requested:.2f}"
        )

def withdraw(account_ref, balance, amount):
    if amount > balance:
        raise InsufficientFundsError(account_ref, balance, amount)
    return balance - amount

try:
    withdraw("ACC-1051", 1000.0, 500)
except InsufficientFundsError as e:
    print(f"Rejected -> {e}")
    print(f"  shortfall: {e.requested - e.balance:.2f}")


**Explanation:** This solution separates validation from orchestration, catches errors at the boundary that is actually able to act on them, and keeps enough context (account reference, field name, or cause) for support and audit follow-up.

### Q52. Chain invalidtransactionerror from valueerror for banking case 3; use sample reference `ACC-1052` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q52: Chain InvalidTransactionError from ValueError (ACC-1052)
class InvalidTransactionError(Exception):
    pass

def process_transaction_amount(raw_amount):
    try:
        amount = float(raw_amount)
        if amount <= 0:
            raise ValueError("amount must be positive")
    except ValueError as exc:
        # 'raise ... from exc' preserves the original ValueError as
        # __cause__, so the traceback still shows the root parsing failure.
        raise InvalidTransactionError(f"[ACC-1052] invalid transaction amount {raw_amount!r}") from exc
    return amount

try:
    process_transaction_amount("100.00")
except InvalidTransactionError as e:
    print(f"{e}")
    print(f"  caused by: {e.__cause__!r}")


**Explanation:** This solution separates validation from orchestration, catches errors at the boundary that is actually able to act on them, and keeps enough context (account reference, field name, or cause) for support and audit follow-up.

### Q53. Process transfers while isolating bad rows for banking case 3; use sample reference `ACC-1053` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q53: Process transfers while isolating bad rows (ACC-1053)
transfers = [
    {"ref": "ACC-1053", "to": "ACC-2001", "amount": 100.0},
    {"ref": "ACC-1053", "to": "ACC-2002", "amount": "bad"},
    {"ref": "ACC-1053", "to": "", "amount": 50.0},
    {"ref": "ACC-1053", "to": "ACC-2003", "amount": -20.0},
]

def validate_transfer(row):
    amount = float(row["amount"])  # ValueError on bad text
    if not row["to"]:
        raise ValueError("missing beneficiary account")
    if amount <= 0:
        raise ValueError("amount must be positive")
    return amount

successes, failures = [], []
for i, row in enumerate(transfers):
    try:
        amount = validate_transfer(row)
        successes.append((i, amount))
    except (ValueError, TypeError) as e:
        # one bad row is isolated and logged; the batch keeps going
        failures.append((i, str(e)))

print(f"[ACC-1053] processed {len(successes)} ok, {len(failures)} failed")
for i, amt in successes:
    print(f"  row {i} OK -> {amt}")
for i, err in failures:
    print(f"  row {i} FAILED -> {err}")


**Explanation:** This solution separates validation from orchestration, catches errors at the boundary that is actually able to act on them, and keeps enough context (account reference, field name, or cause) for support and audit follow-up.

### Q54. Retry a transient payment gateway for banking case 3; use sample reference `ACC-1054` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q54: Retry a transient payment gateway (ACC-1054)
import time

class GatewayTimeoutError(Exception):
    pass

_attempts = {"n": 0}

def call_payment_gateway():
    _attempts["n"] += 1
    if _attempts["n"] <= 0:
        raise GatewayTimeoutError(f"[ACC-1054] gateway timed out (attempt {_attempts['n']})")
    return "PAYMENT-OK"

def call_with_retry(func, max_attempts=3, backoff_seconds=0):
    last_exc = None
    for attempt in range(1, max_attempts + 1):
        try:
            return func()
        except GatewayTimeoutError as e:
            last_exc = e
            print(f"[ACC-1054] attempt {attempt} failed -> {e}")
            time.sleep(backoff_seconds)
    raise GatewayTimeoutError(f"[ACC-1054] gateway unavailable after {max_attempts} attempts") from last_exc

try:
    result = call_with_retry(call_payment_gateway, max_attempts=3)
    print(f"[ACC-1054] result -> {result}")
except GatewayTimeoutError as e:
    print(f"[ACC-1054] gave up -> {e}")


**Explanation:** This solution separates validation from orchestration, catches errors at the boundary that is actually able to act on them, and keeps enough context (account reference, field name, or cause) for support and audit follow-up.

### Q55. Log failed standing instructions for banking case 3; use sample reference `ACC-1055` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q55: Log failed standing instructions (ACC-1055)
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger("standing_instructions")

instructions = [
    {"ref": "ACC-1055", "amount": 200.0},
    {"ref": "ACC-1055", "amount": -50.0},
]

def execute_standing_instruction(instr):
    if instr["amount"] <= 0:
        raise ValueError(f"standing instruction amount must be positive: {instr['amount']}")
    return instr["amount"]

for instr in instructions:
    try:
        amt = execute_standing_instruction(instr)
        logger.info(f"[{instr['ref']}] standing instruction executed: {amt}")
    except ValueError as e:
        # log with context, don't crash the whole batch
        logger.error(f"[{instr['ref']}] standing instruction failed: {e}")


**Explanation:** This solution separates validation from orchestration, catches errors at the boundary that is actually able to act on them, and keeps enough context (account reference, field name, or cause) for support and audit follow-up.

### Q56. Return a structured failure result for banking case 3; use sample reference `ACC-1056` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q56: Return a structured failure result (ACC-1056)
def safe_deposit(account_ref, amount):
    """Instead of letting exceptions propagate to the caller, return a
    structured result. Useful at API/service boundaries where callers
    want a value, not a try/except."""
    try:
        if amount <= 0:
            raise ValueError("deposit amount must be positive")
        return {"ok": True, "account_ref": account_ref, "amount": amount, "error": None}
    except ValueError as e:
        return {"ok": False, "account_ref": account_ref, "amount": None, "error": str(e)}

result = safe_deposit("ACC-1056", -5)
if result["ok"]:
    print(f"[ACC-1056] success -> {result['amount']}")
else:
    print(f"[ACC-1056] failure -> {result['error']}")


**Explanation:** This solution separates validation from orchestration, catches errors at the boundary that is actually able to act on them, and keeps enough context (account reference, field name, or cause) for support and audit follow-up.

### Q57. Validate a nested beneficiary record for banking case 3; use sample reference `ACC-1057` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q57: Validate a nested beneficiary record (ACC-1057)
class KYCValidationError(Exception):
    pass

def validate_beneficiary(record):
    required = ["name", "account", "ifsc"]
    missing = [f for f in required if f not in record or not record[f]]
    if missing:
        raise KYCValidationError(f"[ACC-1057] beneficiary missing fields: {', '.join(missing)}")
    return True

beneficiary = {"name": "Asha Rao", "account": "ACC-9001"}
try:
    validate_beneficiary(beneficiary)
    print(f"[ACC-1057] beneficiary valid -> {beneficiary['name']}")
except KYCValidationError as e:
    print(f"[ACC-1057] -> {e}")


**Explanation:** This solution separates validation from orchestration, catches errors at the boundary that is actually able to act on them, and keeps enough context (account reference, field name, or cause) for support and audit follow-up.

### Q58. Use a context manager for an audit file for banking case 3; use sample reference `ACC-1058` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q58: Use a context manager for an audit file (ACC-1058)
import contextlib, io

@contextlib.contextmanager
def audit_writer(buffer):
    """Guarantees the audit entry is flushed even if the body raises."""
    print(f"[ACC-1058] audit session started")
    try:
        yield buffer
    finally:
        buffer.flush()
        print(f"[ACC-1058] audit session closed, {len(buffer.getvalue())} bytes written")

buf = io.StringIO()
try:
    with audit_writer(buf) as log:
        log.write(f"[ACC-1058] transaction recorded\n")
        raise RuntimeError("downstream notification failed")
except RuntimeError as e:
    print(f"[ACC-1058] handled after cleanup -> {e}")
print(buf.getvalue())


**Explanation:** This solution separates validation from orchestration, catches errors at the boundary that is actually able to act on them, and keeps enough context (account reference, field name, or cause) for support and audit follow-up.

### Q59. Separate validation and recovery for banking case 3; use sample reference `ACC-1059` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q59: Separate validation and recovery (ACC-1059)
class ValidationError(Exception):
    pass

class RecoverableError(Exception):
    pass

def validate_transfer_request(amount, to_account):
    # Validation: caller mistakes -- do NOT retry these, fail fast.
    if amount <= 0:
        raise ValidationError(f"[ACC-1059] amount must be positive")
    if not to_account:
        raise ValidationError(f"[ACC-1059] destination account is required")

def submit_transfer(amount, to_account):
    # Recovery: infrastructure hiccups -- these ARE worth retrying.
    if to_account == "ACC-DOWN":
        raise RecoverableError(f"[ACC-1059] downstream ledger temporarily unavailable")
    return f"TXN-{to_account}-{amount}"

def transfer(amount, to_account, retries=2):
    validate_transfer_request(amount, to_account)  # not retried
    for attempt in range(retries + 1):
        try:
            return submit_transfer(amount, to_account)
        except RecoverableError as e:
            print(f"[ACC-1059] retry {attempt + 1} after -> {e}")
    raise RecoverableError(f"[ACC-1059] transfer failed after {retries} retries")

try:
    print(transfer(100.0, "ACC-2001"))
except ValidationError as e:
    print(f"[ACC-1059] bad request -> {e}")


**Explanation:** This solution separates validation from orchestration, catches errors at the boundary that is actually able to act on them, and keeps enough context (account reference, field name, or cause) for support and audit follow-up.

### Q60. Collect all kyc validation errors for banking case 3; use sample reference `ACC-1060` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q60: Collect all KYC validation errors (ACC-1060)
class KYCValidationError(Exception):
    def __init__(self, errors):
        self.errors = errors
        super().__init__("; ".join(errors))

def validate_kyc(record):
    """Collect every problem instead of stopping at the first, so the
    customer/ops team can fix everything in one pass."""
    errors = []
    if not record.get("pan"):
        errors.append("PAN is required")
    if not record.get("address"):
        errors.append("address is required")
    if not record.get("dob"):
        errors.append("date of birth is required")
    if errors:
        raise KYCValidationError(errors)
    return True

record = {"pan": "ABCDE1234F"} if 3 % 2 else {}
try:
    validate_kyc(record)
    print(f"[ACC-1060] KYC valid")
except KYCValidationError as e:
    print(f"[ACC-1060] KYC failed with {len(e.errors)} issue(s):")
    for err in e.errors:
        print(f"  - {err}")


**Explanation:** This solution separates validation from orchestration, catches errors at the boundary that is actually able to act on them, and keeps enough context (account reference, field name, or cause) for support and audit follow-up.

### Q61. Create insufficientfundserror for banking case 4; use sample reference `ACC-1061` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q61: Create InsufficientFundsError (ACC-1061)
class BankingError(Exception):
    """Base class for all business-rule banking failures."""

class InsufficientFundsError(BankingError):
    def __init__(self, account_ref, balance, requested):
        self.account_ref = account_ref
        self.balance = balance
        self.requested = requested
        super().__init__(
            f"[{account_ref}] insufficient funds: balance={balance:.2f}, requested={requested:.2f}"
        )

def withdraw(account_ref, balance, amount):
    if amount > balance:
        raise InsufficientFundsError(account_ref, balance, amount)
    return balance - amount

try:
    withdraw("ACC-1061", 1000.0, 1500)
except InsufficientFundsError as e:
    print(f"Rejected -> {e}")
    print(f"  shortfall: {e.requested - e.balance:.2f}")


**Explanation:** This solution separates validation from orchestration, catches errors at the boundary that is actually able to act on them, and keeps enough context (account reference, field name, or cause) for support and audit follow-up.

### Q62. Chain invalidtransactionerror from valueerror for banking case 4; use sample reference `ACC-1062` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q62: Chain InvalidTransactionError from ValueError (ACC-1062)
class InvalidTransactionError(Exception):
    pass

def process_transaction_amount(raw_amount):
    try:
        amount = float(raw_amount)
        if amount <= 0:
            raise ValueError("amount must be positive")
    except ValueError as exc:
        # 'raise ... from exc' preserves the original ValueError as
        # __cause__, so the traceback still shows the root parsing failure.
        raise InvalidTransactionError(f"[ACC-1062] invalid transaction amount {raw_amount!r}") from exc
    return amount

try:
    process_transaction_amount("not-a-number")
except InvalidTransactionError as e:
    print(f"{e}")
    print(f"  caused by: {e.__cause__!r}")


**Explanation:** This solution separates validation from orchestration, catches errors at the boundary that is actually able to act on them, and keeps enough context (account reference, field name, or cause) for support and audit follow-up.

### Q63. Process transfers while isolating bad rows for banking case 4; use sample reference `ACC-1063` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q63: Process transfers while isolating bad rows (ACC-1063)
transfers = [
    {"ref": "ACC-1063", "to": "ACC-2001", "amount": 100.0},
    {"ref": "ACC-1063", "to": "ACC-2002", "amount": "bad"},
    {"ref": "ACC-1063", "to": "", "amount": 50.0},
    {"ref": "ACC-1063", "to": "ACC-2003", "amount": -20.0},
]

def validate_transfer(row):
    amount = float(row["amount"])  # ValueError on bad text
    if not row["to"]:
        raise ValueError("missing beneficiary account")
    if amount <= 0:
        raise ValueError("amount must be positive")
    return amount

successes, failures = [], []
for i, row in enumerate(transfers):
    try:
        amount = validate_transfer(row)
        successes.append((i, amount))
    except (ValueError, TypeError) as e:
        # one bad row is isolated and logged; the batch keeps going
        failures.append((i, str(e)))

print(f"[ACC-1063] processed {len(successes)} ok, {len(failures)} failed")
for i, amt in successes:
    print(f"  row {i} OK -> {amt}")
for i, err in failures:
    print(f"  row {i} FAILED -> {err}")


**Explanation:** This solution separates validation from orchestration, catches errors at the boundary that is actually able to act on them, and keeps enough context (account reference, field name, or cause) for support and audit follow-up.

### Q64. Retry a transient payment gateway for banking case 4; use sample reference `ACC-1064` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q64: Retry a transient payment gateway (ACC-1064)
import time

class GatewayTimeoutError(Exception):
    pass

_attempts = {"n": 0}

def call_payment_gateway():
    _attempts["n"] += 1
    if _attempts["n"] <= 1:
        raise GatewayTimeoutError(f"[ACC-1064] gateway timed out (attempt {_attempts['n']})")
    return "PAYMENT-OK"

def call_with_retry(func, max_attempts=3, backoff_seconds=0):
    last_exc = None
    for attempt in range(1, max_attempts + 1):
        try:
            return func()
        except GatewayTimeoutError as e:
            last_exc = e
            print(f"[ACC-1064] attempt {attempt} failed -> {e}")
            time.sleep(backoff_seconds)
    raise GatewayTimeoutError(f"[ACC-1064] gateway unavailable after {max_attempts} attempts") from last_exc

try:
    result = call_with_retry(call_payment_gateway, max_attempts=3)
    print(f"[ACC-1064] result -> {result}")
except GatewayTimeoutError as e:
    print(f"[ACC-1064] gave up -> {e}")


**Explanation:** This solution separates validation from orchestration, catches errors at the boundary that is actually able to act on them, and keeps enough context (account reference, field name, or cause) for support and audit follow-up.

### Q65. Log failed standing instructions for banking case 4; use sample reference `ACC-1065` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q65: Log failed standing instructions (ACC-1065)
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger("standing_instructions")

instructions = [
    {"ref": "ACC-1065", "amount": 200.0},
    {"ref": "ACC-1065", "amount": -50.0},
]

def execute_standing_instruction(instr):
    if instr["amount"] <= 0:
        raise ValueError(f"standing instruction amount must be positive: {instr['amount']}")
    return instr["amount"]

for instr in instructions:
    try:
        amt = execute_standing_instruction(instr)
        logger.info(f"[{instr['ref']}] standing instruction executed: {amt}")
    except ValueError as e:
        # log with context, don't crash the whole batch
        logger.error(f"[{instr['ref']}] standing instruction failed: {e}")


**Explanation:** This solution separates validation from orchestration, catches errors at the boundary that is actually able to act on them, and keeps enough context (account reference, field name, or cause) for support and audit follow-up.

### Q66. Return a structured failure result for banking case 4; use sample reference `ACC-1066` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q66: Return a structured failure result (ACC-1066)
def safe_deposit(account_ref, amount):
    """Instead of letting exceptions propagate to the caller, return a
    structured result. Useful at API/service boundaries where callers
    want a value, not a try/except."""
    try:
        if amount <= 0:
            raise ValueError("deposit amount must be positive")
        return {"ok": True, "account_ref": account_ref, "amount": amount, "error": None}
    except ValueError as e:
        return {"ok": False, "account_ref": account_ref, "amount": None, "error": str(e)}

result = safe_deposit("ACC-1066", 100)
if result["ok"]:
    print(f"[ACC-1066] success -> {result['amount']}")
else:
    print(f"[ACC-1066] failure -> {result['error']}")


**Explanation:** This solution separates validation from orchestration, catches errors at the boundary that is actually able to act on them, and keeps enough context (account reference, field name, or cause) for support and audit follow-up.

### Q67. Validate a nested beneficiary record for banking case 4; use sample reference `ACC-1067` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q67: Validate a nested beneficiary record (ACC-1067)
class KYCValidationError(Exception):
    pass

def validate_beneficiary(record):
    required = ["name", "account", "ifsc"]
    missing = [f for f in required if f not in record or not record[f]]
    if missing:
        raise KYCValidationError(f"[ACC-1067] beneficiary missing fields: {', '.join(missing)}")
    return True

beneficiary = {"name": "Asha Rao", "account": "ACC-9001", "ifsc": "HDFC0001"}
try:
    validate_beneficiary(beneficiary)
    print(f"[ACC-1067] beneficiary valid -> {beneficiary['name']}")
except KYCValidationError as e:
    print(f"[ACC-1067] -> {e}")


**Explanation:** This solution separates validation from orchestration, catches errors at the boundary that is actually able to act on them, and keeps enough context (account reference, field name, or cause) for support and audit follow-up.

### Q68. Use a context manager for an audit file for banking case 4; use sample reference `ACC-1068` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q68: Use a context manager for an audit file (ACC-1068)
import contextlib, io

@contextlib.contextmanager
def audit_writer(buffer):
    """Guarantees the audit entry is flushed even if the body raises."""
    print(f"[ACC-1068] audit session started")
    try:
        yield buffer
    finally:
        buffer.flush()
        print(f"[ACC-1068] audit session closed, {len(buffer.getvalue())} bytes written")

buf = io.StringIO()
try:
    with audit_writer(buf) as log:
        log.write(f"[ACC-1068] transaction recorded\n")
        raise RuntimeError("downstream notification failed")
except RuntimeError as e:
    print(f"[ACC-1068] handled after cleanup -> {e}")
print(buf.getvalue())


**Explanation:** This solution separates validation from orchestration, catches errors at the boundary that is actually able to act on them, and keeps enough context (account reference, field name, or cause) for support and audit follow-up.

### Q69. Separate validation and recovery for banking case 4; use sample reference `ACC-1069` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q69: Separate validation and recovery (ACC-1069)
class ValidationError(Exception):
    pass

class RecoverableError(Exception):
    pass

def validate_transfer_request(amount, to_account):
    # Validation: caller mistakes -- do NOT retry these, fail fast.
    if amount <= 0:
        raise ValidationError(f"[ACC-1069] amount must be positive")
    if not to_account:
        raise ValidationError(f"[ACC-1069] destination account is required")

def submit_transfer(amount, to_account):
    # Recovery: infrastructure hiccups -- these ARE worth retrying.
    if to_account == "ACC-DOWN":
        raise RecoverableError(f"[ACC-1069] downstream ledger temporarily unavailable")
    return f"TXN-{to_account}-{amount}"

def transfer(amount, to_account, retries=2):
    validate_transfer_request(amount, to_account)  # not retried
    for attempt in range(retries + 1):
        try:
            return submit_transfer(amount, to_account)
        except RecoverableError as e:
            print(f"[ACC-1069] retry {attempt + 1} after -> {e}")
    raise RecoverableError(f"[ACC-1069] transfer failed after {retries} retries")

try:
    print(transfer(100.0, "ACC-2001"))
except ValidationError as e:
    print(f"[ACC-1069] bad request -> {e}")


**Explanation:** This solution separates validation from orchestration, catches errors at the boundary that is actually able to act on them, and keeps enough context (account reference, field name, or cause) for support and audit follow-up.

### Q70. Collect all kyc validation errors for banking case 4; use sample reference `ACC-1070` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q70: Collect all KYC validation errors (ACC-1070)
class KYCValidationError(Exception):
    def __init__(self, errors):
        self.errors = errors
        super().__init__("; ".join(errors))

def validate_kyc(record):
    """Collect every problem instead of stopping at the first, so the
    customer/ops team can fix everything in one pass."""
    errors = []
    if not record.get("pan"):
        errors.append("PAN is required")
    if not record.get("address"):
        errors.append("address is required")
    if not record.get("dob"):
        errors.append("date of birth is required")
    if errors:
        raise KYCValidationError(errors)
    return True

record = {"pan": "ABCDE1234F"} if 4 % 2 else {}
try:
    validate_kyc(record)
    print(f"[ACC-1070] KYC valid")
except KYCValidationError as e:
    print(f"[ACC-1070] KYC failed with {len(e.errors)} issue(s):")
    for err in e.errors:
        print(f"  - {err}")


**Explanation:** This solution separates validation from orchestration, catches errors at the boundary that is actually able to act on them, and keeps enough context (account reference, field name, or cause) for support and audit follow-up.


---

## Difficult

Architecture, edge cases, and end-to-end banking systems.


### Q71. Design a banking exception hierarchy for banking case 1; use sample reference `ACC-1071` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q71: Design a banking exception hierarchy (ACC-1071)
class BankingError(Exception):
    """Root of all domain errors -- callers can catch this to handle any
    banking-specific failure without swallowing programming bugs."""

class ValidationError(BankingError):
    """Bad input from the caller (4xx-style)."""

class BusinessRuleError(BankingError):
    """Input was well-formed but violates a banking rule."""

class InsufficientFundsError(BusinessRuleError):
    pass

class AccountNotFoundError(BankingError):
    pass

class InfrastructureError(BankingError):
    """Downstream/technical failure (5xx-style) -- generally retryable."""

class GatewayTimeoutError(InfrastructureError):
    pass


def process(account_ref, amount, accounts):
    if account_ref not in accounts:
        raise AccountNotFoundError(f"[{account_ref}] not found")
    if amount <= 0:
        raise ValidationError(f"[{account_ref}] amount must be positive")
    if amount > accounts[account_ref]:
        raise InsufficientFundsError(f"[{account_ref}] insufficient funds")
    accounts[account_ref] -= amount
    return accounts[account_ref]

accounts = {"ACC-1071": 300.0}
for amt in [50.0, -1.0, 10000.0]:
    try:
        print(f"[ACC-1071] new balance -> {process('ACC-1071', amt, accounts):.2f}")
    except BankingError as e:
        # a single except clause handles every domain error uniformly
        print(f"[ACC-1071] {type(e).__name__} -> {e}")


**Explanation:** The design protects account/ledger state under partial failure, preserves the original cause with `raise ... from`, and makes the failure path explicit rather than silent. In a real system, prints would become structured logs and the in-memory dict would be a transactional store.

### Q72. Make a transfer rollback-safe for banking case 1; use sample reference `ACC-1072` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q72: Make a transfer rollback-safe (ACC-1072)
class TransferError(Exception):
    pass

accounts = {"ACC-1072": 500.0, "ACC-DEST": 100.0}

def debit(acc_ref, amount):
    accounts[acc_ref] -= amount

def credit(acc_ref, amount):
    if acc_ref == "ACC-DEST" and False:
        raise RuntimeError("credit leg failed downstream")
    accounts[acc_ref] += amount

def transfer(src, dst, amount):
    snapshot = dict(accounts)  # protect against partial updates
    try:
        debit(src, amount)
        credit(dst, amount)
    except Exception as exc:
        accounts.clear()
        accounts.update(snapshot)  # rollback both legs
        raise TransferError(f"[{src} -> {dst}] transfer rolled back") from exc

try:
    transfer("ACC-1072", "ACC-DEST", 100.0)
    print(f"[ACC-1072] transfer OK -> {accounts}")
except TransferError as e:
    print(f"{e} (cause: {e.__cause__})")
    print(f"  balances unchanged -> {accounts}")


**Explanation:** The design protects account/ledger state under partial failure, preserves the original cause with `raise ... from`, and makes the failure path explicit rather than silent. In a real system, prints would become structured logs and the in-memory dict would be a transactional store.

### Q73. Preserve causes across service layers for banking case 1; use sample reference `ACC-1073` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q73: Preserve causes across service layers (ACC-1073)
class RepositoryError(Exception):
    pass

class ServiceError(Exception):
    pass

class ApiError(Exception):
    pass

def repository_layer():
    try:
        {}["missing"]
    except KeyError as exc:
        raise RepositoryError(f"[ACC-1073] record lookup failed") from exc

def service_layer():
    try:
        repository_layer()
    except RepositoryError as exc:
        raise ServiceError(f"[ACC-1073] business operation failed") from exc

def api_layer():
    try:
        service_layer()
    except ServiceError as exc:
        raise ApiError(f"[ACC-1073] request could not be completed") from exc

try:
    api_layer()
except ApiError as e:
    # walk the __cause__ chain to see the true root cause end-to-end
    chain = []
    cur = e
    while cur:
        chain.append(f"{type(cur).__name__}: {cur}")
        cur = cur.__cause__
    print(" -> caused by -> ".join(chain))


**Explanation:** The design protects account/ledger state under partial failure, preserves the original cause with `raise ... from`, and makes the failure path explicit rather than silent. In a real system, prints would become structured logs and the in-memory dict would be a transactional store.

### Q74. Implement bounded exponential retry for banking case 1; use sample reference `ACC-1074` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q74: Implement bounded exponential retry (ACC-1074)
import time

class TransientError(Exception):
    pass

_state = {"n": 0}

def flaky_call():
    _state["n"] += 1
    if _state["n"] <= 3:
        raise TransientError(f"[ACC-1074] transient failure #{_state['n']}")
    return "OK"

def retry_with_backoff(func, max_attempts=4, base_delay=0.01):
    for attempt in range(1, max_attempts + 1):
        try:
            return func()
        except TransientError as e:
            if attempt == max_attempts:
                raise TransientError(f"[ACC-1074] exhausted {max_attempts} attempts") from e
            delay = base_delay * (2 ** (attempt - 1))  # exponential backoff
            print(f"[ACC-1074] attempt {attempt} failed ({e}); retrying in {delay:.3f}s")
            time.sleep(delay)

try:
    print(f"[ACC-1074] result -> {retry_with_backoff(flaky_call)}")
except TransientError as e:
    print(f"[ACC-1074] gave up -> {e}")


**Explanation:** The design protects account/ledger state under partial failure, preserves the original cause with `raise ... from`, and makes the failure path explicit rather than silent. In a real system, prints would become structured logs and the in-memory dict would be a transactional store.

### Q75. Create a transaction context manager for banking case 1; use sample reference `ACC-1075` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q75: Create a transaction context manager (ACC-1075)
class TransactionContext:
    """Commits on clean exit, rolls back on any exception -- like a DB
    transaction block. Returning False from __exit__ re-raises the error."""

    def __init__(self, account_ref, ledger):
        self.account_ref = account_ref
        self.ledger = ledger
        self.snapshot = None

    def __enter__(self):
        self.snapshot = dict(self.ledger)
        print(f"[{self.account_ref}] transaction started")
        return self

    def __exit__(self, exc_type, exc, tb):
        if exc_type is None:
            print(f"[{self.account_ref}] transaction committed")
            return False
        self.ledger.clear()
        self.ledger.update(self.snapshot)
        print(f"[{self.account_ref}] transaction rolled back due to {exc_type.__name__}: {exc}")
        return False  # propagate the exception after rollback

ledger = {"ACC-1075": 400.0}
try:
    with TransactionContext("ACC-1075", ledger):
        ledger["ACC-1075"] -= 100.0
        if False:
            raise RuntimeError("posting failed")
except RuntimeError as e:
    print(f"[ACC-1075] -> {e}")
print(f"[ACC-1075] final ledger -> {ledger}")


**Explanation:** The design protects account/ledger state under partial failure, preserves the original cause with `raise ... from`, and makes the failure path explicit rather than silent. In a real system, prints would become structured logs and the in-memory dict would be a transactional store.

### Q76. Build a dead-letter queue for payment events for banking case 1; use sample reference `ACC-1076` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q76: Build a dead-letter queue for payment events (ACC-1076)
dead_letter_queue = []

events = [
    {"ref": "ACC-1076", "amount": 100.0},
    {"ref": "ACC-1076", "amount": "bad-amount"},
    {"ref": "ACC-1076", "amount": -20.0},
]

def process_payment_event(event):
    amount = float(event["amount"])  # ValueError for bad text
    if amount <= 0:
        raise ValueError("amount must be positive")
    return amount

for event in events:
    try:
        amount = process_payment_event(event)
        print(f"[{event['ref']}] processed -> {amount}")
    except (ValueError, TypeError) as e:
        dead_letter_queue.append({"event": event, "error": str(e)})

print(f"[ACC-1076] dead-letter queue has {len(dead_letter_queue)} entr(y/ies):")
for entry in dead_letter_queue:
    print(f"  {entry['event']} -> {entry['error']}")


**Explanation:** The design protects account/ledger state under partial failure, preserves the original cause with `raise ... from`, and makes the failure path explicit rather than silent. In a real system, prints would become structured logs and the in-memory dict would be a transactional store.

### Q77. Make batch settlement failure-isolated for banking case 1; use sample reference `ACC-1077` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q77: Make batch settlement failure-isolated (ACC-1077)
batch = [
    {"ref": "ACC-1077", "amount": 100.0},
    {"ref": "ACC-BAD", "amount": None},
    {"ref": "ACC-1077", "amount": 50.0},
]

def settle(item):
    amount = float(item["amount"])
    return amount

results = {"settled": [], "failed": []}
for item in batch:
    try:
        results["settled"].append((item["ref"], settle(item)))
    except (TypeError, ValueError) as e:
        # one bad item never stops the rest of the batch from settling
        results["failed"].append((item["ref"], str(e)))

print(f"[ACC-1077] settled: {results['settled']}")
print(f"[ACC-1077] failed:  {results['failed']}")
print(f"[ACC-1077] success rate -> {len(results['settled'])}/{len(batch)}")


**Explanation:** The design protects account/ledger state under partial failure, preserves the original cause with `raise ... from`, and makes the failure path explicit rather than silent. In a real system, prints would become structured logs and the in-memory dict would be a transactional store.

### Q78. Report exact paths in nested statements for banking case 1; use sample reference `ACC-1078` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q78: Report exact paths in nested statements (ACC-1078)
def process_batch(rows):
    """Nested try/excepts can obscure WHERE a failure happened. Track the
    step explicitly so error messages point at an exact location."""
    for row_idx, row in enumerate(rows):
        step = "unknown"
        try:
            step = "parse_amount"
            amount = float(row["amount"])
            step = "validate_amount"
            if amount <= 0:
                raise ValueError("amount must be positive")
            step = "lookup_account"
            if row["ref"] != "ACC-1078":
                raise LookupError("account mismatch")
            print(f"[ACC-1078] row {row_idx} OK -> {amount}")
        except (ValueError, LookupError, TypeError) as e:
            print(f"[ACC-1078] row {row_idx} failed at step '{step}' -> {e}")

process_batch([
    {"ref": "ACC-1078", "amount": "100"},
    {"ref": "ACC-1078", "amount": "-5"},
    {"ref": "ACC-OTHER", "amount": "10"},
])


**Explanation:** The design protects account/ledger state under partial failure, preserves the original cause with `raise ... from`, and makes the failure path explicit rather than silent. In a real system, prints would become structured logs and the in-memory dict would be a transactional store.

### Q79. Design idempotent recovery after timeout for banking case 1; use sample reference `ACC-1079` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q79: Design idempotent recovery after timeout (ACC-1079)
processed_idempotency_keys = set()
ledger = {"ACC-1079": 1000.0}

class GatewayTimeoutError(Exception):
    pass

def apply_debit(idempotency_key, account_ref, amount):
    """A retried call after a timeout must not double-apply the debit.
    We check-and-record the idempotency key before mutating state."""
    if idempotency_key in processed_idempotency_keys:
        print(f"[{account_ref}] key {idempotency_key} already applied -- skipping duplicate")
        return ledger[account_ref]
    ledger[account_ref] -= amount
    processed_idempotency_keys.add(idempotency_key)
    return ledger[account_ref]

def debit_with_retry(idempotency_key, account_ref, amount, simulate_timeout_once=[True]):
    try:
        if simulate_timeout_once[0]:
            simulate_timeout_once[0] = False
            raise GatewayTimeoutError(f"[{account_ref}] response lost after commit")
        return apply_debit(idempotency_key, account_ref, amount)
    except GatewayTimeoutError as e:
        print(f"[{account_ref}] timeout ({e}); retrying with same key")
        return apply_debit(idempotency_key, account_ref, amount)

key = "IDEMP-ACC-1079-001"
result = debit_with_retry(key, "ACC-1079", 100.0)
print(f"[ACC-1079] balance after safe retry -> {result:.2f}")


**Explanation:** The design protects account/ledger state under partial failure, preserves the original cause with `raise ... from`, and makes the failure path explicit rather than silent. In a real system, prints would become structured logs and the in-memory dict would be a transactional store.

### Q80. Build an exception-safe reconciliation pipeline for banking case 1; use sample reference `ACC-1080` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q80: Build an exception-safe reconciliation pipeline (ACC-1080)
class ReconciliationError(Exception):
    pass

bank_records = [{"ref": "ACC-1080", "amount": 100.0}, {"ref": "ACC-X", "amount": 50.0}]
ledger_records = [{"ref": "ACC-1080", "amount": 100.0}, {"ref": "ACC-Y", "amount": 75.0}]

def reconcile(bank, ledger):
    report = {"matched": [], "unmatched_bank": [], "unmatched_ledger": [], "errors": []}
    ledger_by_ref = {r["ref"]: r["amount"] for r in ledger}
    for record in bank:
        try:
            ref = record["ref"]
            amount = float(record["amount"])
            if ref in ledger_by_ref and ledger_by_ref[ref] == amount:
                report["matched"].append(ref)
            else:
                report["unmatched_bank"].append(ref)
        except (KeyError, ValueError, TypeError) as e:
            report["errors"].append(str(e))
    bank_refs = {r["ref"] for r in bank}
    report["unmatched_ledger"] = [r["ref"] for r in ledger if r["ref"] not in bank_refs]
    return report

report = reconcile(bank_records, ledger_records)
print(f"[ACC-1080] matched: {report['matched']}")
print(f"[ACC-1080] unmatched (bank only): {report['unmatched_bank']}")
print(f"[ACC-1080] unmatched (ledger only): {report['unmatched_ledger']}")
print(f"[ACC-1080] errors during reconciliation: {report['errors']}")


**Explanation:** The design protects account/ledger state under partial failure, preserves the original cause with `raise ... from`, and makes the failure path explicit rather than silent. In a real system, prints would become structured logs and the in-memory dict would be a transactional store.

### Q81. Design a banking exception hierarchy for banking case 2; use sample reference `ACC-1081` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q81: Design a banking exception hierarchy (ACC-1081)
class BankingError(Exception):
    """Root of all domain errors -- callers can catch this to handle any
    banking-specific failure without swallowing programming bugs."""

class ValidationError(BankingError):
    """Bad input from the caller (4xx-style)."""

class BusinessRuleError(BankingError):
    """Input was well-formed but violates a banking rule."""

class InsufficientFundsError(BusinessRuleError):
    pass

class AccountNotFoundError(BankingError):
    pass

class InfrastructureError(BankingError):
    """Downstream/technical failure (5xx-style) -- generally retryable."""

class GatewayTimeoutError(InfrastructureError):
    pass


def process(account_ref, amount, accounts):
    if account_ref not in accounts:
        raise AccountNotFoundError(f"[{account_ref}] not found")
    if amount <= 0:
        raise ValidationError(f"[{account_ref}] amount must be positive")
    if amount > accounts[account_ref]:
        raise InsufficientFundsError(f"[{account_ref}] insufficient funds")
    accounts[account_ref] -= amount
    return accounts[account_ref]

accounts = {"ACC-1081": 300.0}
for amt in [50.0, -1.0, 10000.0]:
    try:
        print(f"[ACC-1081] new balance -> {process('ACC-1081', amt, accounts):.2f}")
    except BankingError as e:
        # a single except clause handles every domain error uniformly
        print(f"[ACC-1081] {type(e).__name__} -> {e}")


**Explanation:** The design protects account/ledger state under partial failure, preserves the original cause with `raise ... from`, and makes the failure path explicit rather than silent. In a real system, prints would become structured logs and the in-memory dict would be a transactional store.

### Q82. Make a transfer rollback-safe for banking case 2; use sample reference `ACC-1082` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q82: Make a transfer rollback-safe (ACC-1082)
class TransferError(Exception):
    pass

accounts = {"ACC-1082": 500.0, "ACC-DEST": 100.0}

def debit(acc_ref, amount):
    accounts[acc_ref] -= amount

def credit(acc_ref, amount):
    if acc_ref == "ACC-DEST" and True:
        raise RuntimeError("credit leg failed downstream")
    accounts[acc_ref] += amount

def transfer(src, dst, amount):
    snapshot = dict(accounts)  # protect against partial updates
    try:
        debit(src, amount)
        credit(dst, amount)
    except Exception as exc:
        accounts.clear()
        accounts.update(snapshot)  # rollback both legs
        raise TransferError(f"[{src} -> {dst}] transfer rolled back") from exc

try:
    transfer("ACC-1082", "ACC-DEST", 100.0)
    print(f"[ACC-1082] transfer OK -> {accounts}")
except TransferError as e:
    print(f"{e} (cause: {e.__cause__})")
    print(f"  balances unchanged -> {accounts}")


**Explanation:** The design protects account/ledger state under partial failure, preserves the original cause with `raise ... from`, and makes the failure path explicit rather than silent. In a real system, prints would become structured logs and the in-memory dict would be a transactional store.

### Q83. Preserve causes across service layers for banking case 2; use sample reference `ACC-1083` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q83: Preserve causes across service layers (ACC-1083)
class RepositoryError(Exception):
    pass

class ServiceError(Exception):
    pass

class ApiError(Exception):
    pass

def repository_layer():
    try:
        {}["missing"]
    except KeyError as exc:
        raise RepositoryError(f"[ACC-1083] record lookup failed") from exc

def service_layer():
    try:
        repository_layer()
    except RepositoryError as exc:
        raise ServiceError(f"[ACC-1083] business operation failed") from exc

def api_layer():
    try:
        service_layer()
    except ServiceError as exc:
        raise ApiError(f"[ACC-1083] request could not be completed") from exc

try:
    api_layer()
except ApiError as e:
    # walk the __cause__ chain to see the true root cause end-to-end
    chain = []
    cur = e
    while cur:
        chain.append(f"{type(cur).__name__}: {cur}")
        cur = cur.__cause__
    print(" -> caused by -> ".join(chain))


**Explanation:** The design protects account/ledger state under partial failure, preserves the original cause with `raise ... from`, and makes the failure path explicit rather than silent. In a real system, prints would become structured logs and the in-memory dict would be a transactional store.

### Q84. Implement bounded exponential retry for banking case 2; use sample reference `ACC-1084` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q84: Implement bounded exponential retry (ACC-1084)
import time

class TransientError(Exception):
    pass

_state = {"n": 0}

def flaky_call():
    _state["n"] += 1
    if _state["n"] <= 2:
        raise TransientError(f"[ACC-1084] transient failure #{_state['n']}")
    return "OK"

def retry_with_backoff(func, max_attempts=4, base_delay=0.01):
    for attempt in range(1, max_attempts + 1):
        try:
            return func()
        except TransientError as e:
            if attempt == max_attempts:
                raise TransientError(f"[ACC-1084] exhausted {max_attempts} attempts") from e
            delay = base_delay * (2 ** (attempt - 1))  # exponential backoff
            print(f"[ACC-1084] attempt {attempt} failed ({e}); retrying in {delay:.3f}s")
            time.sleep(delay)

try:
    print(f"[ACC-1084] result -> {retry_with_backoff(flaky_call)}")
except TransientError as e:
    print(f"[ACC-1084] gave up -> {e}")


**Explanation:** The design protects account/ledger state under partial failure, preserves the original cause with `raise ... from`, and makes the failure path explicit rather than silent. In a real system, prints would become structured logs and the in-memory dict would be a transactional store.

### Q85. Create a transaction context manager for banking case 2; use sample reference `ACC-1085` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q85: Create a transaction context manager (ACC-1085)
class TransactionContext:
    """Commits on clean exit, rolls back on any exception -- like a DB
    transaction block. Returning False from __exit__ re-raises the error."""

    def __init__(self, account_ref, ledger):
        self.account_ref = account_ref
        self.ledger = ledger
        self.snapshot = None

    def __enter__(self):
        self.snapshot = dict(self.ledger)
        print(f"[{self.account_ref}] transaction started")
        return self

    def __exit__(self, exc_type, exc, tb):
        if exc_type is None:
            print(f"[{self.account_ref}] transaction committed")
            return False
        self.ledger.clear()
        self.ledger.update(self.snapshot)
        print(f"[{self.account_ref}] transaction rolled back due to {exc_type.__name__}: {exc}")
        return False  # propagate the exception after rollback

ledger = {"ACC-1085": 400.0}
try:
    with TransactionContext("ACC-1085", ledger):
        ledger["ACC-1085"] -= 100.0
        if True:
            raise RuntimeError("posting failed")
except RuntimeError as e:
    print(f"[ACC-1085] -> {e}")
print(f"[ACC-1085] final ledger -> {ledger}")


**Explanation:** The design protects account/ledger state under partial failure, preserves the original cause with `raise ... from`, and makes the failure path explicit rather than silent. In a real system, prints would become structured logs and the in-memory dict would be a transactional store.

### Q86. Build a dead-letter queue for payment events for banking case 2; use sample reference `ACC-1086` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q86: Build a dead-letter queue for payment events (ACC-1086)
dead_letter_queue = []

events = [
    {"ref": "ACC-1086", "amount": 100.0},
    {"ref": "ACC-1086", "amount": "bad-amount"},
    {"ref": "ACC-1086", "amount": -20.0},
]

def process_payment_event(event):
    amount = float(event["amount"])  # ValueError for bad text
    if amount <= 0:
        raise ValueError("amount must be positive")
    return amount

for event in events:
    try:
        amount = process_payment_event(event)
        print(f"[{event['ref']}] processed -> {amount}")
    except (ValueError, TypeError) as e:
        dead_letter_queue.append({"event": event, "error": str(e)})

print(f"[ACC-1086] dead-letter queue has {len(dead_letter_queue)} entr(y/ies):")
for entry in dead_letter_queue:
    print(f"  {entry['event']} -> {entry['error']}")


**Explanation:** The design protects account/ledger state under partial failure, preserves the original cause with `raise ... from`, and makes the failure path explicit rather than silent. In a real system, prints would become structured logs and the in-memory dict would be a transactional store.

### Q87. Make batch settlement failure-isolated for banking case 2; use sample reference `ACC-1087` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q87: Make batch settlement failure-isolated (ACC-1087)
batch = [
    {"ref": "ACC-1087", "amount": 100.0},
    {"ref": "ACC-BAD", "amount": None},
    {"ref": "ACC-1087", "amount": 50.0},
]

def settle(item):
    amount = float(item["amount"])
    return amount

results = {"settled": [], "failed": []}
for item in batch:
    try:
        results["settled"].append((item["ref"], settle(item)))
    except (TypeError, ValueError) as e:
        # one bad item never stops the rest of the batch from settling
        results["failed"].append((item["ref"], str(e)))

print(f"[ACC-1087] settled: {results['settled']}")
print(f"[ACC-1087] failed:  {results['failed']}")
print(f"[ACC-1087] success rate -> {len(results['settled'])}/{len(batch)}")


**Explanation:** The design protects account/ledger state under partial failure, preserves the original cause with `raise ... from`, and makes the failure path explicit rather than silent. In a real system, prints would become structured logs and the in-memory dict would be a transactional store.

### Q88. Report exact paths in nested statements for banking case 2; use sample reference `ACC-1088` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q88: Report exact paths in nested statements (ACC-1088)
def process_batch(rows):
    """Nested try/excepts can obscure WHERE a failure happened. Track the
    step explicitly so error messages point at an exact location."""
    for row_idx, row in enumerate(rows):
        step = "unknown"
        try:
            step = "parse_amount"
            amount = float(row["amount"])
            step = "validate_amount"
            if amount <= 0:
                raise ValueError("amount must be positive")
            step = "lookup_account"
            if row["ref"] != "ACC-1088":
                raise LookupError("account mismatch")
            print(f"[ACC-1088] row {row_idx} OK -> {amount}")
        except (ValueError, LookupError, TypeError) as e:
            print(f"[ACC-1088] row {row_idx} failed at step '{step}' -> {e}")

process_batch([
    {"ref": "ACC-1088", "amount": "100"},
    {"ref": "ACC-1088", "amount": "-5"},
    {"ref": "ACC-OTHER", "amount": "10"},
])


**Explanation:** The design protects account/ledger state under partial failure, preserves the original cause with `raise ... from`, and makes the failure path explicit rather than silent. In a real system, prints would become structured logs and the in-memory dict would be a transactional store.

### Q89. Design idempotent recovery after timeout for banking case 2; use sample reference `ACC-1089` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q89: Design idempotent recovery after timeout (ACC-1089)
processed_idempotency_keys = set()
ledger = {"ACC-1089": 1000.0}

class GatewayTimeoutError(Exception):
    pass

def apply_debit(idempotency_key, account_ref, amount):
    """A retried call after a timeout must not double-apply the debit.
    We check-and-record the idempotency key before mutating state."""
    if idempotency_key in processed_idempotency_keys:
        print(f"[{account_ref}] key {idempotency_key} already applied -- skipping duplicate")
        return ledger[account_ref]
    ledger[account_ref] -= amount
    processed_idempotency_keys.add(idempotency_key)
    return ledger[account_ref]

def debit_with_retry(idempotency_key, account_ref, amount, simulate_timeout_once=[True]):
    try:
        if simulate_timeout_once[0]:
            simulate_timeout_once[0] = False
            raise GatewayTimeoutError(f"[{account_ref}] response lost after commit")
        return apply_debit(idempotency_key, account_ref, amount)
    except GatewayTimeoutError as e:
        print(f"[{account_ref}] timeout ({e}); retrying with same key")
        return apply_debit(idempotency_key, account_ref, amount)

key = "IDEMP-ACC-1089-001"
result = debit_with_retry(key, "ACC-1089", 100.0)
print(f"[ACC-1089] balance after safe retry -> {result:.2f}")


**Explanation:** The design protects account/ledger state under partial failure, preserves the original cause with `raise ... from`, and makes the failure path explicit rather than silent. In a real system, prints would become structured logs and the in-memory dict would be a transactional store.

### Q90. Build an exception-safe reconciliation pipeline for banking case 2; use sample reference `ACC-1090` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q90: Build an exception-safe reconciliation pipeline (ACC-1090)
class ReconciliationError(Exception):
    pass

bank_records = [{"ref": "ACC-1090", "amount": 100.0}, {"ref": "ACC-X", "amount": 50.0}]
ledger_records = [{"ref": "ACC-1090", "amount": 100.0}, {"ref": "ACC-Y", "amount": 75.0}]

def reconcile(bank, ledger):
    report = {"matched": [], "unmatched_bank": [], "unmatched_ledger": [], "errors": []}
    ledger_by_ref = {r["ref"]: r["amount"] for r in ledger}
    for record in bank:
        try:
            ref = record["ref"]
            amount = float(record["amount"])
            if ref in ledger_by_ref and ledger_by_ref[ref] == amount:
                report["matched"].append(ref)
            else:
                report["unmatched_bank"].append(ref)
        except (KeyError, ValueError, TypeError) as e:
            report["errors"].append(str(e))
    bank_refs = {r["ref"] for r in bank}
    report["unmatched_ledger"] = [r["ref"] for r in ledger if r["ref"] not in bank_refs]
    return report

report = reconcile(bank_records, ledger_records)
print(f"[ACC-1090] matched: {report['matched']}")
print(f"[ACC-1090] unmatched (bank only): {report['unmatched_bank']}")
print(f"[ACC-1090] unmatched (ledger only): {report['unmatched_ledger']}")
print(f"[ACC-1090] errors during reconciliation: {report['errors']}")


**Explanation:** The design protects account/ledger state under partial failure, preserves the original cause with `raise ... from`, and makes the failure path explicit rather than silent. In a real system, prints would become structured logs and the in-memory dict would be a transactional store.

### Q91. Design a banking exception hierarchy for banking case 3; use sample reference `ACC-1091` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q91: Design a banking exception hierarchy (ACC-1091)
class BankingError(Exception):
    """Root of all domain errors -- callers can catch this to handle any
    banking-specific failure without swallowing programming bugs."""

class ValidationError(BankingError):
    """Bad input from the caller (4xx-style)."""

class BusinessRuleError(BankingError):
    """Input was well-formed but violates a banking rule."""

class InsufficientFundsError(BusinessRuleError):
    pass

class AccountNotFoundError(BankingError):
    pass

class InfrastructureError(BankingError):
    """Downstream/technical failure (5xx-style) -- generally retryable."""

class GatewayTimeoutError(InfrastructureError):
    pass


def process(account_ref, amount, accounts):
    if account_ref not in accounts:
        raise AccountNotFoundError(f"[{account_ref}] not found")
    if amount <= 0:
        raise ValidationError(f"[{account_ref}] amount must be positive")
    if amount > accounts[account_ref]:
        raise InsufficientFundsError(f"[{account_ref}] insufficient funds")
    accounts[account_ref] -= amount
    return accounts[account_ref]

accounts = {"ACC-1091": 300.0}
for amt in [50.0, -1.0, 10000.0]:
    try:
        print(f"[ACC-1091] new balance -> {process('ACC-1091', amt, accounts):.2f}")
    except BankingError as e:
        # a single except clause handles every domain error uniformly
        print(f"[ACC-1091] {type(e).__name__} -> {e}")


**Explanation:** The design protects account/ledger state under partial failure, preserves the original cause with `raise ... from`, and makes the failure path explicit rather than silent. In a real system, prints would become structured logs and the in-memory dict would be a transactional store.

### Q92. Make a transfer rollback-safe for banking case 3; use sample reference `ACC-1092` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q92: Make a transfer rollback-safe (ACC-1092)
class TransferError(Exception):
    pass

accounts = {"ACC-1092": 500.0, "ACC-DEST": 100.0}

def debit(acc_ref, amount):
    accounts[acc_ref] -= amount

def credit(acc_ref, amount):
    if acc_ref == "ACC-DEST" and False:
        raise RuntimeError("credit leg failed downstream")
    accounts[acc_ref] += amount

def transfer(src, dst, amount):
    snapshot = dict(accounts)  # protect against partial updates
    try:
        debit(src, amount)
        credit(dst, amount)
    except Exception as exc:
        accounts.clear()
        accounts.update(snapshot)  # rollback both legs
        raise TransferError(f"[{src} -> {dst}] transfer rolled back") from exc

try:
    transfer("ACC-1092", "ACC-DEST", 100.0)
    print(f"[ACC-1092] transfer OK -> {accounts}")
except TransferError as e:
    print(f"{e} (cause: {e.__cause__})")
    print(f"  balances unchanged -> {accounts}")


**Explanation:** The design protects account/ledger state under partial failure, preserves the original cause with `raise ... from`, and makes the failure path explicit rather than silent. In a real system, prints would become structured logs and the in-memory dict would be a transactional store.

### Q93. Preserve causes across service layers for banking case 3; use sample reference `ACC-1093` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q93: Preserve causes across service layers (ACC-1093)
class RepositoryError(Exception):
    pass

class ServiceError(Exception):
    pass

class ApiError(Exception):
    pass

def repository_layer():
    try:
        {}["missing"]
    except KeyError as exc:
        raise RepositoryError(f"[ACC-1093] record lookup failed") from exc

def service_layer():
    try:
        repository_layer()
    except RepositoryError as exc:
        raise ServiceError(f"[ACC-1093] business operation failed") from exc

def api_layer():
    try:
        service_layer()
    except ServiceError as exc:
        raise ApiError(f"[ACC-1093] request could not be completed") from exc

try:
    api_layer()
except ApiError as e:
    # walk the __cause__ chain to see the true root cause end-to-end
    chain = []
    cur = e
    while cur:
        chain.append(f"{type(cur).__name__}: {cur}")
        cur = cur.__cause__
    print(" -> caused by -> ".join(chain))


**Explanation:** The design protects account/ledger state under partial failure, preserves the original cause with `raise ... from`, and makes the failure path explicit rather than silent. In a real system, prints would become structured logs and the in-memory dict would be a transactional store.

### Q94. Implement bounded exponential retry for banking case 3; use sample reference `ACC-1094` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q94: Implement bounded exponential retry (ACC-1094)
import time

class TransientError(Exception):
    pass

_state = {"n": 0}

def flaky_call():
    _state["n"] += 1
    if _state["n"] <= 3:
        raise TransientError(f"[ACC-1094] transient failure #{_state['n']}")
    return "OK"

def retry_with_backoff(func, max_attempts=4, base_delay=0.01):
    for attempt in range(1, max_attempts + 1):
        try:
            return func()
        except TransientError as e:
            if attempt == max_attempts:
                raise TransientError(f"[ACC-1094] exhausted {max_attempts} attempts") from e
            delay = base_delay * (2 ** (attempt - 1))  # exponential backoff
            print(f"[ACC-1094] attempt {attempt} failed ({e}); retrying in {delay:.3f}s")
            time.sleep(delay)

try:
    print(f"[ACC-1094] result -> {retry_with_backoff(flaky_call)}")
except TransientError as e:
    print(f"[ACC-1094] gave up -> {e}")


**Explanation:** The design protects account/ledger state under partial failure, preserves the original cause with `raise ... from`, and makes the failure path explicit rather than silent. In a real system, prints would become structured logs and the in-memory dict would be a transactional store.

### Q95. Create a transaction context manager for banking case 3; use sample reference `ACC-1095` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q95: Create a transaction context manager (ACC-1095)
class TransactionContext:
    """Commits on clean exit, rolls back on any exception -- like a DB
    transaction block. Returning False from __exit__ re-raises the error."""

    def __init__(self, account_ref, ledger):
        self.account_ref = account_ref
        self.ledger = ledger
        self.snapshot = None

    def __enter__(self):
        self.snapshot = dict(self.ledger)
        print(f"[{self.account_ref}] transaction started")
        return self

    def __exit__(self, exc_type, exc, tb):
        if exc_type is None:
            print(f"[{self.account_ref}] transaction committed")
            return False
        self.ledger.clear()
        self.ledger.update(self.snapshot)
        print(f"[{self.account_ref}] transaction rolled back due to {exc_type.__name__}: {exc}")
        return False  # propagate the exception after rollback

ledger = {"ACC-1095": 400.0}
try:
    with TransactionContext("ACC-1095", ledger):
        ledger["ACC-1095"] -= 100.0
        if False:
            raise RuntimeError("posting failed")
except RuntimeError as e:
    print(f"[ACC-1095] -> {e}")
print(f"[ACC-1095] final ledger -> {ledger}")


**Explanation:** The design protects account/ledger state under partial failure, preserves the original cause with `raise ... from`, and makes the failure path explicit rather than silent. In a real system, prints would become structured logs and the in-memory dict would be a transactional store.

### Q96. Build a dead-letter queue for payment events for banking case 3; use sample reference `ACC-1096` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q96: Build a dead-letter queue for payment events (ACC-1096)
dead_letter_queue = []

events = [
    {"ref": "ACC-1096", "amount": 100.0},
    {"ref": "ACC-1096", "amount": "bad-amount"},
    {"ref": "ACC-1096", "amount": -20.0},
]

def process_payment_event(event):
    amount = float(event["amount"])  # ValueError for bad text
    if amount <= 0:
        raise ValueError("amount must be positive")
    return amount

for event in events:
    try:
        amount = process_payment_event(event)
        print(f"[{event['ref']}] processed -> {amount}")
    except (ValueError, TypeError) as e:
        dead_letter_queue.append({"event": event, "error": str(e)})

print(f"[ACC-1096] dead-letter queue has {len(dead_letter_queue)} entr(y/ies):")
for entry in dead_letter_queue:
    print(f"  {entry['event']} -> {entry['error']}")


**Explanation:** The design protects account/ledger state under partial failure, preserves the original cause with `raise ... from`, and makes the failure path explicit rather than silent. In a real system, prints would become structured logs and the in-memory dict would be a transactional store.

### Q97. Make batch settlement failure-isolated for banking case 3; use sample reference `ACC-1097` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q97: Make batch settlement failure-isolated (ACC-1097)
batch = [
    {"ref": "ACC-1097", "amount": 100.0},
    {"ref": "ACC-BAD", "amount": None},
    {"ref": "ACC-1097", "amount": 50.0},
]

def settle(item):
    amount = float(item["amount"])
    return amount

results = {"settled": [], "failed": []}
for item in batch:
    try:
        results["settled"].append((item["ref"], settle(item)))
    except (TypeError, ValueError) as e:
        # one bad item never stops the rest of the batch from settling
        results["failed"].append((item["ref"], str(e)))

print(f"[ACC-1097] settled: {results['settled']}")
print(f"[ACC-1097] failed:  {results['failed']}")
print(f"[ACC-1097] success rate -> {len(results['settled'])}/{len(batch)}")


**Explanation:** The design protects account/ledger state under partial failure, preserves the original cause with `raise ... from`, and makes the failure path explicit rather than silent. In a real system, prints would become structured logs and the in-memory dict would be a transactional store.

### Q98. Report exact paths in nested statements for banking case 3; use sample reference `ACC-1098` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q98: Report exact paths in nested statements (ACC-1098)
def process_batch(rows):
    """Nested try/excepts can obscure WHERE a failure happened. Track the
    step explicitly so error messages point at an exact location."""
    for row_idx, row in enumerate(rows):
        step = "unknown"
        try:
            step = "parse_amount"
            amount = float(row["amount"])
            step = "validate_amount"
            if amount <= 0:
                raise ValueError("amount must be positive")
            step = "lookup_account"
            if row["ref"] != "ACC-1098":
                raise LookupError("account mismatch")
            print(f"[ACC-1098] row {row_idx} OK -> {amount}")
        except (ValueError, LookupError, TypeError) as e:
            print(f"[ACC-1098] row {row_idx} failed at step '{step}' -> {e}")

process_batch([
    {"ref": "ACC-1098", "amount": "100"},
    {"ref": "ACC-1098", "amount": "-5"},
    {"ref": "ACC-OTHER", "amount": "10"},
])


**Explanation:** The design protects account/ledger state under partial failure, preserves the original cause with `raise ... from`, and makes the failure path explicit rather than silent. In a real system, prints would become structured logs and the in-memory dict would be a transactional store.

### Q99. Design idempotent recovery after timeout for banking case 3; use sample reference `ACC-1099` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q99: Design idempotent recovery after timeout (ACC-1099)
processed_idempotency_keys = set()
ledger = {"ACC-1099": 1000.0}

class GatewayTimeoutError(Exception):
    pass

def apply_debit(idempotency_key, account_ref, amount):
    """A retried call after a timeout must not double-apply the debit.
    We check-and-record the idempotency key before mutating state."""
    if idempotency_key in processed_idempotency_keys:
        print(f"[{account_ref}] key {idempotency_key} already applied -- skipping duplicate")
        return ledger[account_ref]
    ledger[account_ref] -= amount
    processed_idempotency_keys.add(idempotency_key)
    return ledger[account_ref]

def debit_with_retry(idempotency_key, account_ref, amount, simulate_timeout_once=[True]):
    try:
        if simulate_timeout_once[0]:
            simulate_timeout_once[0] = False
            raise GatewayTimeoutError(f"[{account_ref}] response lost after commit")
        return apply_debit(idempotency_key, account_ref, amount)
    except GatewayTimeoutError as e:
        print(f"[{account_ref}] timeout ({e}); retrying with same key")
        return apply_debit(idempotency_key, account_ref, amount)

key = "IDEMP-ACC-1099-001"
result = debit_with_retry(key, "ACC-1099", 100.0)
print(f"[ACC-1099] balance after safe retry -> {result:.2f}")


**Explanation:** The design protects account/ledger state under partial failure, preserves the original cause with `raise ... from`, and makes the failure path explicit rather than silent. In a real system, prints would become structured logs and the in-memory dict would be a transactional store.

### Q100. Build an exception-safe reconciliation pipeline for banking case 3; use sample reference `ACC-1100` and explain the result.

**Hints**

1. Identify the smallest risky statement and the narrow exception it can raise.
2. Keep business-rule failures distinct from programming or infrastructure failures.
3. For medium/difficult work, preserve the original cause and protect partially updated balances.

**Edge cases to test:** empty input, invalid types, boundary values, and a realistic failure path.

In [ ]:
# Q100: Build an exception-safe reconciliation pipeline (ACC-1100)
class ReconciliationError(Exception):
    pass

bank_records = [{"ref": "ACC-1100", "amount": 100.0}, {"ref": "ACC-X", "amount": 50.0}]
ledger_records = [{"ref": "ACC-1100", "amount": 100.0}, {"ref": "ACC-Y", "amount": 75.0}]

def reconcile(bank, ledger):
    report = {"matched": [], "unmatched_bank": [], "unmatched_ledger": [], "errors": []}
    ledger_by_ref = {r["ref"]: r["amount"] for r in ledger}
    for record in bank:
        try:
            ref = record["ref"]
            amount = float(record["amount"])
            if ref in ledger_by_ref and ledger_by_ref[ref] == amount:
                report["matched"].append(ref)
            else:
                report["unmatched_bank"].append(ref)
        except (KeyError, ValueError, TypeError) as e:
            report["errors"].append(str(e))
    bank_refs = {r["ref"] for r in bank}
    report["unmatched_ledger"] = [r["ref"] for r in ledger if r["ref"] not in bank_refs]
    return report

report = reconcile(bank_records, ledger_records)
print(f"[ACC-1100] matched: {report['matched']}")
print(f"[ACC-1100] unmatched (bank only): {report['unmatched_bank']}")
print(f"[ACC-1100] unmatched (ledger only): {report['unmatched_ledger']}")
print(f"[ACC-1100] errors during reconciliation: {report['errors']}")


**Explanation:** The design protects account/ledger state under partial failure, preserves the original cause with `raise ... from`, and makes the failure path explicit rather than silent. In a real system, prints would become structured logs and the in-memory dict would be a transactional store.